# BERTopic NLP Pipeline

This is the main modeling notebook for the final Mood-Based Watch Recommender. It transforms the cleaned/enriched title dataset into the production NLP catalog used by the deployed FastAPI backend.

The goal is not simple title-to-title similarity. The final product needs a richer representation of every movie/show so it can respond to natural prompts such as `I feel sad and heartbroken but want something joyous and comedic`, `I want a cute movie but not animation`, or `I want something like The Terminator`.

This notebook builds and reviews:

1. enriched modeling text for every title
2. sentence-embedding semantic features
3. BERTopic-style topic neighborhoods and human-readable topic labels
4. low-confidence topic flags instead of hiding difficult assignments
5. weighted mood, tone, theme, conflict, pace, emotional arc, and setting facets
6. prompt-aware hybrid recommendation scoring
7. stress tests for exclusions, contradictions, typos, imperfect English, and current-emotion versus desired-content prompts
8. final CSV/JSON-ready features used by the web app backend

The original cleaned dataset is not overwritten. Each major output is written as a separate reviewable artifact so the modeling choices can be defended during grading.


In [1]:
from pathlib import Path
import hashlib
import json
import math
import re
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from IPython.display import display

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
from bertopic.dimensionality import BaseDimensionalityReduction
from bertopic.vectorizers import ClassTfidfTransformer

pd.set_option("display.max_colwidth", 220)
pd.set_option("display.max_columns", 40)


## Phase 0: Configuration

Use the Phase 2 enriched dataset when available. The enriched file keeps long plot text in separate columns, so the original cleaned descriptions remain intact.

In [2]:
DATA_DIR = Path("data")
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

BASE_DATA_PATH = DATA_DIR / "cleaned_watch_data.csv"
ENRICHED_DATA_PATH = DATA_DIR / "cleaned_watch_data_phase2_enriched.csv"

CORPUS_OUTPUT_PATH = DATA_DIR / "nlp_modeling_corpus.csv"
EMBEDDINGS_OUTPUT_PATH = DATA_DIR / "title_semantic_embeddings.npy"
EMBEDDING_INDEX_OUTPUT_PATH = DATA_DIR / "title_embedding_index.csv"
EMBEDDING_MODEL_INFO_PATH = MODELS_DIR / "embedding_model_info.json"

BERTOPIC_MODEL_DIR = MODELS_DIR / "bertopic_watch_model"
TOPIC_ASSIGNMENTS_PATH = DATA_DIR / "bertopic_title_topics.csv"
TOPIC_INFO_PATH = DATA_DIR / "bertopic_topic_info.csv"
TOPIC_KEYWORDS_PATH = DATA_DIR / "bertopic_topic_keywords.csv"
TOPIC_SAMPLES_PATH = DATA_DIR / "bertopic_topic_samples.csv"
TOPIC_REVIEW_PATH = DATA_DIR / "bertopic_topic_review.csv"
FACETS_OUTPUT_PATH = DATA_DIR / "title_facet_labels.csv"
FINAL_FEATURES_OUTPUT_PATH = DATA_DIR / "final_nlp_recommender_features.csv"
PROMPT_EVALUATION_PATH = DATA_DIR / "prompt_recommendation_evaluation.csv"

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
RANDOM_STATE = 42

# Set this to a smaller number while experimenting, for example 1000.
# Keep None for the full dataset.
MAX_ROWS = None

# Long plots are helpful, but very long full plot sections can dominate topic modeling.
MAX_DESCRIPTION_WORDS = 120
MAX_EXTENDED_PLOT_WORDS = 180
BERTOPIC_PIPELINE_VERSION = "4.0-intent-aware-topic-facet-review"
BERTOPIC_MODEL_INFO_PATH = MODELS_DIR / "bertopic_model_info.json"
TOPIC_STOP_WORDS = sorted(set(ENGLISH_STOP_WORDS) | {
    "story", "plot", "expanded", "format", "genre", "genres", "movie",
    "tvseries", "tvminiseries", "tv", "series", "nan", "based", "classified", "strongest", "signals",
})
BATCH_SIZE = 64


## Phase 1: Load Data And Build Modeling Text

The modeling text is structured so BERTopic sees genre, content type, short description, and any verified long plot text.

Important: titles are not over-weighted. The title helps trace results, but the semantic model should learn from premise/theme/tone, not just title words.

In [3]:
def word_count(text):
    return len(re.findall(r"\b\w+\b", str(text)))


def truncate_words(text, max_words):
    words = str(text).split()
    if len(words) <= max_words:
        return str(text)
    return " ".join(words[:max_words])


def clean_for_modeling(text):
    if pd.isna(text):
        return ""
    text = str(text).replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def source_text_for_modeling(row):
    description = truncate_words(clean_for_modeling(row.get("description", "")), MAX_DESCRIPTION_WORDS)
    extended = clean_for_modeling(row.get("source_enrichment_text", ""))
    if not extended and bool(row.get("has_source_enrichment", False)):
        extended = clean_for_modeling(row.get("enrichment_text", ""))
    if extended:
        extended = truncate_words(extended, MAX_EXTENDED_PLOT_WORDS)
        return f"Story: {description} Expanded plot: {extended}"
    return f"Story: {description}"


def build_model_text(row):
    genres = str(row.get("genres", "")).replace(",", ", ")
    content_type = str(row.get("content_type", ""))
    description = source_text_for_modeling(row)

    parts = [
        f"Format: {content_type}.",
        f"Genres: {genres}.",
        description,
    ]

    return " ".join(parts)


def build_semantic_text(row):
    genres = str(row.get("genres", "")).replace(",", ", ")
    description = source_text_for_modeling(row)
    content_type = str(row.get("content_type", ""))
    return " ".join([f"Format: {content_type}.", f"Genres: {genres}.", description])


data_path = ENRICHED_DATA_PATH if ENRICHED_DATA_PATH.exists() else BASE_DATA_PATH
watch_df = pd.read_csv(data_path).reset_index(drop=True)

if MAX_ROWS is not None:
    watch_df = watch_df.head(MAX_ROWS).copy()

watch_df["row_id"] = np.arange(len(watch_df))
watch_df["has_phase2_enrichment"] = watch_df.get("has_phase2_enrichment", False).fillna(False).astype(bool)
watch_df["description_word_count"] = watch_df["description"].fillna("").map(word_count)
watch_df["enrichment_word_count"] = watch_df.get("enrichment_word_count", pd.Series(np.nan, index=watch_df.index))
watch_df["model_text"] = watch_df.apply(build_model_text, axis=1)
watch_df["semantic_text"] = watch_df.apply(build_semantic_text, axis=1)
watch_df["model_text_word_count"] = watch_df["model_text"].map(word_count)

corpus_columns = [
    "row_id", "tconst", "content_type", "primary_title", "display_title", "original_title",
    "release_year", "genres", "average_rating", "num_votes", "weighted_rating",
    "description", "description_word_count", "has_phase2_enrichment", "enrichment_word_count",
    "wikipedia_url", "source_section", "model_text", "semantic_text", "model_text_word_count",
]
corpus_columns = [column for column in corpus_columns if column in watch_df.columns]
watch_df[corpus_columns].to_csv(CORPUS_OUTPUT_PATH, index=False)

print("Loaded:", data_path)
print("Rows:", len(watch_df))
print("Phase 2 enriched rows:", int(watch_df["has_phase2_enrichment"].sum()))
print("Median model-text words:", watch_df["model_text_word_count"].median())
print("Saved corpus:", CORPUS_OUTPUT_PATH)

display(watch_df[["tconst", "primary_title", "release_year", "genres", "description_word_count", "has_phase2_enrichment", "model_text_word_count"]].head(10))


Loaded: data/cleaned_watch_data_phase2_enriched.csv
Rows: 7848
Phase 2 enriched rows: 446
Median model-text words: 127.0
Saved corpus: data/nlp_modeling_corpus.csv


,tconst,primary_title,release_year,genres,description_word_count,has_phase2_enrichment,model_text_word_count
0,tt0102926,The Silence of the Lambs,1991,"Crime,Drama,Thriller",220,True,307
1,tt0103064,Terminator 2: Judgment Day,1991,"Action,Sci-Fi",223,True,317
2,tt0110357,The Lion King,1994,"Adventure,Animation,Drama",209,True,303
3,tt0110912,Pulp Fiction,1994,"Crime,Drama",218,True,308
4,tt0111161,The Shawshank Redemption,1994,Drama,199,True,310
5,tt0120338,Titanic,1997,"Drama,Romance",133,False,128
6,tt0121164,Corpse Bride,2005,"Animation,Drama,Family",215,True,307
7,tt0172495,Gladiator,2000,"Action,Adventure,Drama",198,True,297
8,tt0212720,A.I. Artificial Intelligence,2001,"Drama,Sci-Fi",223,True,316
9,tt0217930,Egypt Uncovered,1998,Documentary,94,False,99


## Phase 2: Generate Sentence Embeddings

These embeddings are the semantic backbone of the system. BERTopic will use them for clustering, and the hybrid recommender will use them for title/query similarity.

In [4]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
texts = watch_df["semantic_text"].fillna("").tolist()
semantic_text_fingerprint = hashlib.sha256("\n".join(texts).encode("utf-8")).hexdigest()

rebuild_embeddings = True
if EMBEDDINGS_OUTPUT_PATH.exists() and EMBEDDING_INDEX_OUTPUT_PATH.exists() and EMBEDDING_MODEL_INFO_PATH.exists():
    existing_index = pd.read_csv(EMBEDDING_INDEX_OUTPUT_PATH)
    existing_info = json.loads(EMBEDDING_MODEL_INFO_PATH.read_text())
    same_rows = len(existing_index) == len(watch_df) and existing_index["tconst"].tolist() == watch_df["tconst"].tolist()
    same_text = existing_info.get("semantic_text_fingerprint") == semantic_text_fingerprint
    same_model = existing_info.get("embedding_model_name") == EMBEDDING_MODEL_NAME
    same_version = existing_info.get("pipeline_version") == BERTOPIC_PIPELINE_VERSION

    if same_rows and same_text and same_model and same_version:
        embeddings = np.load(EMBEDDINGS_OUTPUT_PATH)
        rebuild_embeddings = False
        print("Loaded existing embeddings:", embeddings.shape)
    else:
        print("Existing embeddings are stale or do not match current text. Rebuilding.")

if rebuild_embeddings:
    embeddings = embedding_model.encode(
        texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    np.save(EMBEDDINGS_OUTPUT_PATH, embeddings)

embedding_index = watch_df[["row_id", "tconst", "primary_title", "release_year", "genres", "content_type"]].copy()
embedding_index.to_csv(EMBEDDING_INDEX_OUTPUT_PATH, index=False)

EMBEDDING_MODEL_INFO_PATH.write_text(json.dumps({
    "embedding_model_name": EMBEDDING_MODEL_NAME,
    "embedding_shape": list(embeddings.shape),
    "normalized": True,
    "text_column": "semantic_text",
    "semantic_text_fingerprint": semantic_text_fingerprint,
    "pipeline_version": BERTOPIC_PIPELINE_VERSION,
}, indent=2))

print("Embedding shape:", embeddings.shape)
print("Saved:", EMBEDDINGS_OUTPUT_PATH)
print("Saved:", EMBEDDING_INDEX_OUTPUT_PATH)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded existing embeddings: (7848, 384)
Embedding shape: (7848, 384)
Saved: data/title_semantic_embeddings.npy
Saved: data/title_embedding_index.csv


## Phase 3: Fit BERTopic

BERTopic discovers semantic clusters, then names them with topic keywords. Outlier topic `-1` means HDBSCAN did not confidently place that title into a cluster.

In [5]:
saved_model_is_current = False
if BERTOPIC_MODEL_DIR.exists() and TOPIC_ASSIGNMENTS_PATH.exists() and BERTOPIC_MODEL_INFO_PATH.exists():
    saved_assignments = pd.read_csv(TOPIC_ASSIGNMENTS_PATH)
    saved_model_info = json.loads(BERTOPIC_MODEL_INFO_PATH.read_text())
    saved_model_is_current = (
        len(saved_assignments) == len(watch_df)
        and saved_assignments["tconst"].tolist() == watch_df["tconst"].tolist()
        and saved_model_info.get("pipeline_version") == BERTOPIC_PIPELINE_VERSION
        and saved_model_info.get("semantic_text_fingerprint") == semantic_text_fingerprint
    )

if saved_model_is_current:
    topic_model = BERTopic.load(str(BERTOPIC_MODEL_DIR))
    for column in ["bertopic_topic", "bertopic_probability", "bertopic_was_outlier", "bertopic_assignment_confidence"]:
        watch_df[column] = saved_assignments[column].to_numpy()
    topics = watch_df["bertopic_topic"].tolist()
    probabilities = None
    print("Loaded current BERTopic model and reduced topic assignments.")
else:
    vectorizer_model = CountVectorizer(
        stop_words=TOPIC_STOP_WORDS,
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.65,
    )

    umap_model = BaseDimensionalityReduction()

    hdbscan_model = MiniBatchKMeans(
        n_clusters=20,
        batch_size=1024,
        n_init=20,
        random_state=RANDOM_STATE,
    )

    ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

    topic_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        ctfidf_model=ctfidf_model,
        calculate_probabilities=False,
        verbose=True,
    )

    raw_topics, probabilities = topic_model.fit_transform(watch_df["model_text"].tolist(), embeddings)
    raw_topics = np.asarray(raw_topics)
    watch_df["bertopic_was_outlier"] = raw_topics == -1
    topics = raw_topics.copy()
    if np.any(raw_topics == -1):
        topics = np.asarray(topic_model.reduce_outliers(
            watch_df["model_text"].tolist(),
            raw_topics.tolist(),
            strategy="embeddings",
            embeddings=embeddings,
            threshold=0.0,
        ))
        topic_model.update_topics(
            watch_df["model_text"].tolist(),
            topics=topics.tolist(),
            vectorizer_model=vectorizer_model,
            ctfidf_model=ctfidf_model,
        )
    watch_df["bertopic_topic"] = topics

    topic_centroids = {}
    for topic_id in sorted(set(topics)):
        centroid = embeddings[topics == topic_id].mean(axis=0)
        topic_centroids[topic_id] = centroid / max(np.linalg.norm(centroid), 1e-12)
    centroid_confidence = np.asarray([
        float(np.clip(embeddings[row_id] @ topic_centroids[topic_id], 0.0, 1.0))
        for row_id, topic_id in enumerate(topics)
    ])
    if probabilities is not None:
        raw_probability = probabilities.max(axis=1)
        assignment_confidence = np.where(raw_topics == -1, centroid_confidence, np.maximum(raw_probability, centroid_confidence))
    else:
        assignment_confidence = centroid_confidence
    watch_df["bertopic_assignment_confidence"] = assignment_confidence
    watch_df["bertopic_probability"] = assignment_confidence

topic_sizes = watch_df["bertopic_topic"].value_counts().to_dict()
watch_df["bertopic_topic_size"] = watch_df["bertopic_topic"].map(topic_sizes).astype(int)
confidence_floor = max(0.18, float(watch_df["bertopic_assignment_confidence"].quantile(0.08)))
watch_df["bertopic_low_confidence"] = (
    watch_df["bertopic_assignment_confidence"].lt(confidence_floor)
    | watch_df["bertopic_topic_size"].lt(25)
    | watch_df["bertopic_topic"].eq(-1)
)

print("Discovered topics:", len(set(topics)))
print("Original HDBSCAN outliers:", int(watch_df["bertopic_was_outlier"].sum()))
print("Remaining outliers after reduction:", int((watch_df["bertopic_topic"] == -1).sum()))
print("Low-confidence topic assignments flagged:", int(watch_df["bertopic_low_confidence"].sum()))
print("Topic confidence floor:", round(confidence_floor, 3))


Loaded current BERTopic model and reduced topic assignments.
Discovered topics: 20
Original HDBSCAN outliers: 0
Remaining outliers after reduction: 0
Low-confidence topic assignments flagged: 640
Topic confidence floor: 0.436


## Phase 4: Save Topic Assignments, Keywords, And Review Tables

These files are meant for inspection. Topic names should be manually reviewed before being used in a user-facing product.

In [6]:
topic_info = topic_model.get_topic_info()

keyword_rows = []
for topic_id in sorted(set(topics)):
    if topic_id == -1:
        continue
    for rank, (word, score) in enumerate(topic_model.get_topic(topic_id) or [], start=1):
        keyword_rows.append({"topic_id": topic_id, "rank": rank, "keyword": word, "score": score})

topic_keywords = pd.DataFrame(keyword_rows)

def keyword_blob(topic_id):
    return " ".join(
        topic_keywords[topic_keywords["topic_id"] == topic_id]
        .sort_values("rank")
        .head(12)["keyword"]
        .fillna("")
        .astype(str)
        .tolist()
    ).casefold()


def topic_genre_set(value):
    return {item.strip().casefold() for item in str(value).split(",") if item.strip()}


def dominant_genres_for_topic(topic_id, n=5):
    genre_counts = Counter()
    subset = watch_df[watch_df["bertopic_topic"] == topic_id]
    for value in subset["genres"].fillna(""):
        genre_counts.update(topic_genre_set(value))
    return [genre.title() if genre != "sci-fi" else "Sci-Fi" for genre, _ in genre_counts.most_common(n)]


def assign_topic_label(topic_id):
    if topic_id == -1:
        return "Unassigned or Low-Signal Outliers"
    subset = watch_df[watch_df["bertopic_topic"] == topic_id]
    if len(subset) < 25:
        return "Low-Support Mixed Cluster for Manual Review"
    blob = keyword_blob(topic_id)
    dominant = dominant_genres_for_topic(topic_id, n=8)
    first_genre = dominant[0].casefold() if dominant else ""
    genres = {genre.casefold() for genre in dominant}

    keyword_rules = [
        (("hindi" in blob or "tamil" in blob or "telugu" in blob or "indian" in blob or "khan" in blob), "Indian Cinema and Regional Language Stories"),
        (("south korean" in blob or "chinese" in blob or "japanese" in blob or "manga" in blob or "anime" in blob), "East Asian Cinema, Anime, and Serialized Action"),
        (("spanish" in blob or "foreign language" in blob or "danish" in blob or "françois" in blob or "pedro" in blob), "International and Auteur Drama"),
        (("bible" in blob or "downton" in blob or "abbey" in blob or "history romance" in blob), "Historical, Period, and Faith-Based Drama"),
        (("detective" in blob or "police procedural" in blob or "procedural" in blob or "television crime" in blob), "Crime, Investigation, and Procedural Stories"),
        (("thriller combination" in blob or "premise action" in blob or "cop" in blob or "lawbreaking" in blob), "Thriller, Crime, and High-Stakes Action"),
        (("sports" in blob or "football" in blob or "coach" in blob or "boxer" in blob or "athletic" in blob or "sport combination" in blob), "Sports, Competition, and Physical Challenge"),
        (("documentary music" in blob or "songwriter" in blob or "album" in blob or "concert" in blob), "Music, Performance, and Artist Journeys"),
        (("romantic" in blob or "romance" in blob or "wedding" in blob or "situations love" in blob), "Romance, Relationships, and Romantic Comedy"),
        (("sitcom" in blob or "television sitcom" in blob or "american sitcom" in blob), "Sitcoms, Comedy Ensembles, and Light TV"),
        (("horror" in blob or "terror" in blob or "psychological horror" in blob), "Horror, Supernatural Threats, and Dark Suspense"),
        (("star trek" in blob or "science fiction" in blob or "fiction television" in blob or "mystery sci" in blob), "Science Fiction, Technology, and Speculative Worlds"),
        (("marvel" in blob or "superhero" in blob or "cartoon" in blob or "animated television" in blob or "walt disney" in blob), "Animation, Superheroes, and Family Adventure"),
        (("christmas" in blob or "santa" in blob or "holiday" in blob), "Holiday, Family, and Comfort Viewing"),
        (("premise documentary" in blob or "documentary combination" in blob or "biography documentary" in blob or first_genre == "documentary"), "Documentary, Reality, and True-Life Stories"),
    ]
    for matched, label in keyword_rules:
        if matched:
            return label

    if first_genre == "comedy":
        return "Comedy, Social Friction, and Light Entertainment"
    if first_genre == "animation":
        return "Animation, Family, and Coming-of-Age Adventure"
    if first_genre == "documentary":
        return "Documentary, Reality, and True-Life Stories"
    if first_genre == "sport":
        return "Sports, Competition, and Physical Challenge"
    if first_genre == "crime":
        return "Crime, Investigation, and Procedural Stories"
    if first_genre == "action" or "adventure" in genres:
        return "Action, Adventure, and High-Stakes Conflict"
    if first_genre == "drama" and "romance" in genres:
        return "Drama, Romance, and Character Relationships"
    if genres:
        return " / ".join(dominant[:3]) + " Cluster"
    return "Mixed Narrative Cluster"


review_rows = []
for topic_id in sorted(watch_df["bertopic_topic"].unique()):
    subset = watch_df[watch_df["bertopic_topic"] == topic_id]
    keywords = topic_keywords[topic_keywords["topic_id"] == topic_id].sort_values("rank").head(10)["keyword"].tolist()
    dominant_genres = dominant_genres_for_topic(topic_id)
    avg_confidence = float(subset["bertopic_assignment_confidence"].mean())
    low_confidence_share = float(subset["bertopic_low_confidence"].mean())
    topic_quality = "strong"
    if topic_id == -1 or len(subset) < 25 or low_confidence_share > 0.35:
        topic_quality = "needs_review"
    elif avg_confidence < 0.27 or low_confidence_share > 0.18:
        topic_quality = "watchlist"
    review_rows.append({
        "topic_id": int(topic_id),
        "topic_label": assign_topic_label(topic_id),
        "title_count": int(len(subset)),
        "avg_assignment_confidence": round(avg_confidence, 4),
        "low_confidence_share": round(low_confidence_share, 4),
        "topic_quality": topic_quality,
        "dominant_genres": ", ".join(dominant_genres),
        "top_keywords": ", ".join(keywords),
        "sample_titles": "; ".join(subset.sort_values(["bertopic_assignment_confidence", "weighted_rating"], ascending=False).head(6)["primary_title"].tolist()),
    })

topic_review = pd.DataFrame(review_rows).sort_values(["topic_quality", "title_count"], ascending=[True, False])
topic_label_map = topic_review.set_index("topic_id")["topic_label"].to_dict()
topic_quality_map = topic_review.set_index("topic_id")["topic_quality"].to_dict()
watch_df["bertopic_topic_label"] = watch_df["bertopic_topic"].map(topic_label_map)
watch_df["bertopic_topic_quality"] = watch_df["bertopic_topic"].map(topic_quality_map)

topic_info = topic_info.merge(
    topic_review.rename(columns={"topic_id": "Topic"}),
    on="Topic",
    how="left",
)
topic_info.to_csv(TOPIC_INFO_PATH, index=False)
topic_keywords.to_csv(TOPIC_KEYWORDS_PATH, index=False)
topic_review.to_csv(TOPIC_REVIEW_PATH, index=False)

assignment_columns = [
    "row_id", "tconst", "primary_title", "release_year", "content_type", "genres",
    "average_rating", "num_votes", "weighted_rating", "bertopic_topic", "bertopic_topic_label",
    "bertopic_topic_quality", "bertopic_topic_size", "bertopic_probability",
    "bertopic_was_outlier", "bertopic_assignment_confidence", "bertopic_low_confidence",
    "description", "has_phase2_enrichment", "enrichment_word_count",
]
assignment_columns = [column for column in assignment_columns if column in watch_df.columns]
watch_df[assignment_columns].to_csv(TOPIC_ASSIGNMENTS_PATH, index=False)

sample_rows = []
for topic_id in topic_review[topic_review["topic_id"] != -1]["topic_id"].head(80):
    topic_titles = watch_df[watch_df["bertopic_topic"] == topic_id].sort_values(
        ["bertopic_assignment_confidence", "weighted_rating"], ascending=False
    ).head(8)
    keywords = ", ".join(topic_keywords[topic_keywords["topic_id"] == topic_id].head(8)["keyword"].tolist())
    for _, row in topic_titles.iterrows():
        sample_rows.append({
            "topic_id": topic_id,
            "topic_label": row.get("bertopic_topic_label", ""),
            "topic_quality": row.get("bertopic_topic_quality", ""),
            "topic_keywords": keywords,
            "tconst": row["tconst"],
            "primary_title": row["primary_title"],
            "release_year": row["release_year"],
            "genres": row["genres"],
            "assignment_confidence": row["bertopic_assignment_confidence"],
            "description": row["description"],
        })

topic_samples = pd.DataFrame(sample_rows)
topic_samples.to_csv(TOPIC_SAMPLES_PATH, index=False)

# Pickle keeps the model local and reloadable for this project environment.
topic_model.save(str(BERTOPIC_MODEL_DIR), serialization="pickle")
BERTOPIC_MODEL_INFO_PATH.write_text(json.dumps({
    "pipeline_version": BERTOPIC_PIPELINE_VERSION,
    "semantic_text_fingerprint": semantic_text_fingerprint,
    "topic_count": len(set(topics)),
    "remaining_outliers": int((watch_df["bertopic_topic"] == -1).sum()),
    "original_outliers": int(watch_df["bertopic_was_outlier"].sum()),
    "low_confidence_assignments": int(watch_df["bertopic_low_confidence"].sum()),
    "confidence_floor": round(float(confidence_floor), 4),
    "cluster_backend": "MiniBatchKMeans",
    "cluster_parameters": {"n_clusters": 20, "batch_size": 1024, "n_init": 20},
    "dimensionality_reduction": "skipped",
}, indent=2))

print("Saved topic info:", TOPIC_INFO_PATH)
print("Saved topic keywords:", TOPIC_KEYWORDS_PATH)
print("Saved topic assignments:", TOPIC_ASSIGNMENTS_PATH)
print("Saved topic review:", TOPIC_REVIEW_PATH)
print("Saved topic samples:", TOPIC_SAMPLES_PATH)
print("Saved BERTopic model:", BERTOPIC_MODEL_DIR)

display(topic_review)


2026-07-07 22:35:33,284 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


Saved topic info: data/bertopic_topic_info.csv
Saved topic keywords: data/bertopic_topic_keywords.csv
Saved topic assignments: data/bertopic_title_topics.csv
Saved topic review: data/bertopic_topic_review.csv
Saved topic samples: data/bertopic_topic_samples.csv
Saved BERTopic model: models/bertopic_watch_model


,topic_id,topic_label,title_count,avg_assignment_confidence,low_confidence_share,topic_quality,dominant_genres,top_keywords,sample_titles
19,19,Low-Support Mixed Cluster for Manual Review,12,0.6788,1.0000,needs_review,"Drama, Crime, Romance, Thriller, Music","glass, ifc, butterfly, jen, cracks, harwood, shattered, breakdown, cheung, notes",Fracture; Broken; Cracks; Breaking News; Broken; Shattered
0,0,"Crime, Investigation, and Procedural Stories",644,0.6017,0.0093,strong,"Crime, Drama, Mystery, Thriller, Action","television crime, detective, procedural, american crime, crime documentary, police procedural, thriller television, cbs, itv, crimes",Catching Killers; True Detective; Unsolved Mysteries; The Code; The Break; American Crime
1,1,Indian Cinema and Regional Language Stories,615,0.6281,0.0049,strong,"Drama, Action, Comedy, Crime, Romance","hindi, hindi language, indian hindi, tamil, khan, telugu, kapoor, raj, tamil language, indian tamil",In Your Name; Sanam Teri Kasam; Vettaiyaadu Vilaiyaadu; Chakravyuh; Malikappuram; Writer Padmabhushan
2,2,"Romance, Relationships, and Romantic Comedy",533,0.5510,0.0300,strong,"Drama, Romance, Comedy, Biography, Music","american romantic, romantic comedy, comedy romance, situations love, music romance, romance combination, comedy film, wedding, beauty, american coming","Crazy, Stupid, Love.; Things You Can Tell Just by Looking at Her; The Names of Love; Love Today; Her; Stuck in Love."
3,3,"Thriller, Crime, and High-Stakes Action",491,0.6122,0.0061,strong,"Drama, Crime, Thriller, Action, Mystery","thriller combination, premise action, premise crime, pieces lawbreaking, compromise suspense, cop, crime documentary, material crime, material action, mystery combination",AKA; Yeh Saali Zindagi; Sethupathi; Maayavan; Section 375; Naandhi
4,4,"Sitcoms, Comedy Ensembles, and Light TV",490,0.5278,0.0939,strong,"Comedy, Drama, Romance, Animation, Crime","sitcom, sitcom created, television sitcom, comedy television, talk, abc, american television, american sitcom, hosted, aired american",Flack; The Guest Book; As If; Getting On; Episodes; Selfie
5,5,"Horror, Supernatural Threats, and Dark Suspense",478,0.5550,0.0293,strong,"Drama, Thriller, Mystery, Horror, Crime","horror film, horror mystery, horror thriller, thriller film, combination fear, terror secrets, action horror, psychological horror, mystery sci, horror sci",Saw; In Search of Darkness; Insidious; Prisoners; Sinister; Get Out
6,6,"Romance, Relationships, and Romantic Comedy",469,0.6169,0.0021,strong,"Drama, Comedy, Romance, Crime, Thriller","romance combination, material drama, asserted drama, tvmovie drama, settings drama, family combination, premise biography, tvepisode, asserted comedy, music combination",Set It Up; What Will People Say; Someone I Used to Know; Intimacy; The Half of It; The Days
7,7,"Science Fiction, Technology, and Speculative Worlds",467,0.5468,0.0664,strong,"Drama, Action, Sci-Fi, Mystery, Adventure","renewed, season premiered, second season, star trek, trek, final season, channel, fiction television, mystery sci, game reality",The Event; Emergence; Crisis; Explained; From; Ascension
8,8,"Thriller, Crime, and High-Stakes Action",465,0.5043,0.1484,strong,"Drama, Crime, Biography, Action, Comedy","thriller film, skin, jesse, significance lawbreaking, harry potter, biographical crime, damon, potter, downey, jury","Roman J. Israel, Esq.; Widows; The Report; Road to Perdition; Stranger Than Fiction; American Gangster"


## Phase 5: Create Weighted Multi-Label Facets

BERTopic gives broad semantic neighborhoods. Facets make the product more interpretable and more controllable. Each title receives weighted labels for mood, tone, theme, conflict, pace, emotional arc, and setting.

These facets are deliberately more detailed than simple keywords. They include positive, negative, genre-specific, and user-language variants so the backend can understand everyday prompts like `cute`, `cozy`, `dark`, `tense`, `thought-provoking`, `children's movie`, or `not romantic`.

Why this matters for the final website:

- Prompt parsing can map user wording to the same labels used to describe titles.
- Exclusion prompts can be handled more safely because unwanted genres/facets are visible.
- Recommendation explanations can say why a title matched without exposing technical internals.
- Ambiguous prompts can still produce useful results because scoring is based on several weighted signals rather than one exact keyword.


In [7]:
FACET_DEFINITIONS = {
    "mood": {
        "comforting": "warm cozy gentle wholesome safe comforting heartfelt soft kind friendship family belonging healing reassurance low stress",
        "joyful": "joyful cheerful happy bright fun celebratory playful delightful optimistic feel good good mood smiles laughter",
        "sad": "sad grief loss heartbreak illness death regret emotional tragedy tearful melancholy sorrow mourning lonely painful",
        "tense": "tense suspenseful anxious dangerous threat survival paranoia pressure high stakes edge of seat urgent dread",
        "romantic": "romantic love longing intimacy relationship chemistry passion tenderness dating marriage heartbreak desire affection",
        "funny": "funny comedic witty absurd playful silly satire humorous lighthearted laugh hilarious jokes banter comedy",
        "dark": "dark bleak disturbing violent grim trauma corruption cruelty psychological unsettling cynical morally harsh",
        "hopeful": "hopeful inspiring resilient uplifting redemption courage healing second chances optimism perseverance recovery meaningful",
        "adventurous": "adventurous journey quest exploration discovery action danger excitement travel heroes mission treasure",
        "mind_bending": "mind bending surreal reality memory dreams identity time perception mystery twist puzzle psychological strange",
        "inspiring": "motivating inspirational ambitious triumph persistence underdog achievement courage discipline growth success",
    },
    "tone": {
        "gentle": "gentle soft calm tender patient quiet warm low intensity soothing relaxed nonviolent",
        "gritty": "gritty realistic harsh crime street violence raw morally compromised tough bleak urban",
        "whimsical": "whimsical imaginative quirky magical playful charming colorful strange eccentric fairytale",
        "sincere": "sincere heartfelt honest emotional intimate character driven earnest tender grounded",
        "satirical": "satirical social critique absurd ironic class politics hypocrisy comedy biting witty",
        "bleak": "bleak hopeless tragic oppressive despair isolation suffering grim nihilistic heavy",
        "suspenseful": "suspenseful mystery secrets investigation danger twists tension thriller uncertain",
        "epic": "epic grand sweeping historical battle destiny scale heroic spectacle large world",
    },
    "themes": {
        "family": "family parents children siblings marriage home generational bonds care conflict reconciliation",
        "friendship": "friendship loyalty companionship found family support trust belonging team connection",
        "identity": "identity self discovery transformation belonging gender memory personal truth reinvention",
        "revenge": "revenge vengeance payback retaliation justice obsession violence retribution",
        "class_conflict": "class conflict inequality wealth poverty social hierarchy privilege exploitation workers elite",
        "survival": "survival escape disaster outbreak war wilderness apocalypse danger endurance rescue",
        "grief": "grief mourning loss death memory sorrow healing trauma bereavement acceptance",
        "ambition": "ambition success fame power competition career obsession status performance goals",
        "morality": "morality ethics guilt corruption justice crime consequences impossible choices conscience",
        "love": "love romance longing desire marriage heartbreak intimacy connection devotion",
        "healing": "healing recovery forgiveness second chances therapy resilience grief repair emotional support",
        "coming_of_age": "coming of age youth adolescence growing up school first love self discovery maturity",
    },
    "conflict_type": {
        "internal": "internal conflict guilt trauma fear identity memory grief personal struggle self doubt",
        "interpersonal": "interpersonal conflict family relationship betrayal friendship rivalry romance argument",
        "societal": "societal conflict class politics injustice oppression discrimination institutions community",
        "criminal": "criminal conflict murder police investigation heist gangster corruption law detective",
        "survival": "survival conflict escape threat disaster outbreak war hunted trapped attack",
        "supernatural": "supernatural conflict ghosts demons magic monsters curse paranormal fantasy creature",
        "competition": "competition contest sport rivalry performance tournament auditions games challenge",
    },
    "pace": {
        "slow_burn": "slow burn gradual quiet contemplative patient atmospheric character study reflective",
        "steady": "steady balanced unfolding drama investigation emotional development measured",
        "fast_paced": "fast paced action urgent chase rapid danger energetic momentum",
        "chaotic": "chaotic frantic absurd explosive unpredictable wild escalating messy intense",
    },
    "emotional_arc": {
        "uplifting": "uplifting hopeful inspiring healing redemption joyful satisfying feel good happy ending",
        "tragic": "tragic devastating loss death doomed sorrow suffering heartbreak painful ending",
        "bittersweet": "bittersweet mixed emotions tender sadness hope memory acceptance wistful",
        "cathartic": "cathartic release justice closure revenge confession emotional breakthrough relief",
        "unsettling": "unsettling ambiguous disturbing eerie uncomfortable dread unresolved haunting",
        "escapist": "escapist fun entertaining light immersive adventure fantasy comedy distraction relaxation",
    },
    "setting": {
        "war": "war battlefield soldiers occupation military resistance conflict historical combat",
        "school": "school students teachers campus teenage classroom education coming of age",
        "workplace": "workplace office career colleagues industry ambition professional job business",
        "prison": "prison jail incarceration inmates guards confinement escape sentence",
        "urban_crime": "urban crime city police gangs streets corruption investigation detective",
        "small_town": "small town community secrets neighbors rural local mystery village",
        "fantasy_world": "fantasy world magic kingdom mythical creatures prophecy adventure enchanted",
        "space_scifi": "space science fiction aliens future technology robots dystopia cyberpunk",
        "music_stage": "music stage concert band singer performer theater dance audition",
        "home_family": "home family household parents children domestic neighborhood everyday life",
    },
}

facet_label_rows = []
facet_vectors = {}

for category, labels in FACET_DEFINITIONS.items():
    label_names = list(labels.keys())
    label_texts = [labels[name] for name in label_names]
    label_embeddings = embedding_model.encode(
        label_texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    similarities = embeddings @ label_embeddings.T
    facet_vectors[category] = (label_names, similarities)

    for row_idx, row in watch_df.iterrows():
        scores = similarities[row_idx]
        top_indices = np.argsort(scores)[::-1][:4]
        for rank, label_idx in enumerate(top_indices, start=1):
            facet_label_rows.append({
                "row_id": row["row_id"],
                "tconst": row["tconst"],
                "primary_title": row["primary_title"],
                "release_year": row["release_year"],
                "facet_category": category,
                "facet_label": label_names[label_idx],
                "facet_score": float(scores[label_idx]),
                "rank": rank,
            })

facet_labels = pd.DataFrame(facet_label_rows)
facet_labels.to_csv(FACETS_OUTPUT_PATH, index=False)

print("Saved facet labels:", FACETS_OUTPUT_PATH)
display(facet_labels.head(30))


Saved facet labels: data/title_facet_labels.csv


,row_id,tconst,primary_title,release_year,facet_category,facet_label,facet_score,rank
0,0,tt0102926,The Silence of the Lambs,1991,mood,dark,0.310405,1
1,0,tt0102926,The Silence of the Lambs,1991,mood,funny,0.175009,2
2,0,tt0102926,The Silence of the Lambs,1991,mood,mind_bending,0.145190,3
3,0,tt0102926,The Silence of the Lambs,1991,mood,tense,0.144508,4
4,1,tt0103064,Terminator 2: Judgment Day,1991,mood,dark,0.202079,1
5,1,tt0103064,Terminator 2: Judgment Day,1991,mood,tense,0.193571,2
6,1,tt0103064,Terminator 2: Judgment Day,1991,mood,mind_bending,0.155750,3
7,1,tt0103064,Terminator 2: Judgment Day,1991,mood,funny,0.137100,4
8,2,tt0110357,The Lion King,1994,mood,inspiring,0.210312,1
9,2,tt0110357,The Lion King,1994,mood,dark,0.192229,2


## Phase 6: Final NLP Feature Table

This table connects the original title metadata to its topic assignment and top facet labels. It is the main feature table for the next recommender notebook.

In [8]:
def top_facets_for_title(row_id, category, n=3):
    subset = facet_labels[(facet_labels["row_id"] == row_id) & (facet_labels["facet_category"] == category)]
    subset = subset.sort_values("rank").head(n)
    return "; ".join(f"{r.facet_label}:{r.facet_score:.3f}" for r in subset.itertuples())

feature_df = watch_df[[
    "row_id", "tconst", "content_type", "primary_title", "display_title", "original_title",
    "release_year", "genres", "average_rating", "num_votes", "weighted_rating",
    "description", "has_phase2_enrichment", "enrichment_word_count",
    "bertopic_topic", "bertopic_topic_label", "bertopic_topic_quality", "bertopic_topic_size",
    "bertopic_probability", "bertopic_was_outlier", "bertopic_assignment_confidence", "bertopic_low_confidence",
]].copy()

for category in FACET_DEFINITIONS:
    feature_df[f"top_{category}_facets"] = feature_df["row_id"].apply(lambda row_id, c=category: top_facets_for_title(row_id, c, 3))

feature_df.to_csv(FINAL_FEATURES_OUTPUT_PATH, index=False)

print("Saved final NLP feature table:", FINAL_FEATURES_OUTPUT_PATH)
display(feature_df.head(10))


Saved final NLP feature table: data/final_nlp_recommender_features.csv


,row_id,tconst,content_type,primary_title,display_title,original_title,release_year,genres,average_rating,num_votes,weighted_rating,description,has_phase2_enrichment,enrichment_word_count,bertopic_topic,bertopic_topic_label,bertopic_topic_quality,bertopic_topic_size,bertopic_probability,bertopic_was_outlier,bertopic_assignment_confidence,bertopic_low_confidence,top_mood_facets,top_tone_facets,top_themes_facets,top_conflict_type_facets,top_pace_facets,top_emotional_arc_facets,top_setting_facets
0,0,tt0102926,movie,The Silence of the Lambs,The Silence of the Lambs,The Silence of the Lambs,1991,"Crime,Drama,Thriller",8.6,1473918,8.554,"Clarice Starling , a young FBI trainee at Quantico , is recruited by Behavioral Science Unit chief Jack Crawford to interview Dr. Hannibal Lecter , a brilliant and cannibalistic psychiatrist imprisoned at the Baltimo...",True,733,8,"Thriller, Crime, and High-Stakes Action",strong,465,0.406205,False,0.406205,True,dark:0.310; funny:0.175; mind_bending:0.145,gritty:0.227; suspenseful:0.182; gentle:0.168,morality:0.163; revenge:0.162; ambition:0.072,criminal:0.310; supernatural:0.165; internal:0.068,chaotic:0.184; steady:0.100; slow_burn:0.058,unsettling:0.171; escapist:0.107; cathartic:0.104,urban_crime:0.245; prison:0.161; space_scifi:0.132
1,1,tt0103064,movie,Terminator 2: Judgment Day,Terminator 2: Judgment Day,Terminator 2: Judgment Day,1991,"Action,Sci-Fi",8.6,1128166,8.541,"In 2029, Earth has been ravaged by the war between the malevolent artificial intelligence Skynet and the human resistance. Skynet sends the T-1000 —an advanced, shape-shifting prototype Terminator made of virtually i...",True,658,8,"Thriller, Crime, and High-Stakes Action",strong,465,0.394549,False,0.394549,True,dark:0.202; tense:0.194; mind_bending:0.156,gritty:0.199; suspenseful:0.160; satirical:0.125,revenge:0.287; survival:0.199; healing:0.115,survival:0.177; criminal:0.104; internal:0.101,chaotic:0.150; fast_paced:0.078; steady:0.072,tragic:0.154; cathartic:0.139; bittersweet:0.104,space_scifi:0.362; prison:0.135; war:0.085
2,2,tt0110357,movie,The Lion King,The Lion King 3D,The Lion King,1994,"Adventure,Animation,Drama",8.5,1090882,8.444,"In the Pride Lands, a pride of lions rules over the kingdom from Pride Rock. King Mufasa and Queen Sarabi 's newborn son, Simba , is presented to the gathered animals by Rafiki , the mandrill who serves as the kingdo...",True,663,13,"East Asian Cinema, Anime, and Serialized Action",strong,372,0.324552,False,0.324552,True,inspiring:0.210; dark:0.192; hopeful:0.175,epic:0.245; bleak:0.227; suspenseful:0.178,morality:0.240; revenge:0.227; ambition:0.187,societal:0.219; supernatural:0.217; interpersonal:0.167,steady:0.181; chaotic:0.179; slow_burn:0.094,cathartic:0.178; uplifting:0.169; unsettling:0.165,fantasy_world:0.343; war:0.099; small_town:0.069
3,3,tt0110912,movie,Pulp Fiction,Pulp Fiction,Pulp Fiction,1994,"Crime,Drama",8.9,2118762,8.860,"A young Butch Coolidge is visited by Captain Koons, an Air Force pilot. Koons reveals a gold watch to Butch and explains that the watch is a family heirloom and belonged to Butch's father, who served with Koons and d...",True,623,16,"Sports, Competition, and Physical Challenge",strong,264,0.388651,False,0.388651,True,adventurous:0.164; dark:0.154; tense:0.148,suspenseful:0.245; gritty:0.129; satirical:0.120,revenge:0.193; friendship:0.168; morality:0.099,criminal:0.265; competition:0.165; interpersonal:0.149,fast_paced:0.117; chaotic:0.108; slow_burn:0.097,escapist:0.173; tragic:0.128; cathartic:0.127,war:0.199; space_scifi:0.166; urban_crime:0.151
4,4,tt0111161,movie,The Shawshank Redemption,The Shawshank Redemption,The Shawshank Redemption,1994,Drama,9.3,2759621,9.261,"In 1947, Portland, Maine , banker Andy Dufresne arrives at Shawshank State Prison to serve two consecutive life sentences for murdering his wife and her lover. He is befriended by Ellis Boyd ""Red"" Redding, a contraba...",True,704,11,"Sitcoms, Comedy Ensembles, and Light TV",stron

## Phase 7: Prompt-Aware Hybrid NLP Recommender

This production-oriented ranking layer supports both title-seed recommendations and free-text mood recommendations. It extracts prompt intent, detects requested format, infers genres and moods, applies hard exclusions, weights relevant facets, and calibrates user-facing match scores against the eligible catalog.

The final backend implementation in `backend/recommender.py` extends this notebook logic with additional deployment safeguards: title lookup, movie/TV filtering from prompt wording, broad descriptor support, child-safe versus soft/cute handling, and hard genre exclusions such as `no animation`, `not horror`, `without romance`, and `no drama`.


In [9]:
def normalize_title(value):
    return re.sub(r"\s+", " ", str(value).lower()).strip()


def find_title(query, df=watch_df, top_n=10):
    query_norm = normalize_title(query)
    title_columns = ["primary_title", "display_title", "original_title"]
    matches = df.copy()
    exact_mask = pd.Series(False, index=matches.index)
    contains_mask = pd.Series(False, index=matches.index)

    for column in title_columns:
        normalized = matches[column].map(normalize_title)
        exact_mask = exact_mask | normalized.eq(query_norm)
        contains_mask = contains_mask | normalized.str.contains(re.escape(query_norm), regex=True, na=False)

    result = matches[exact_mask | contains_mask].copy()
    result["match_rank"] = np.where(exact_mask[exact_mask | contains_mask], 0, 1)
    return result.sort_values(["match_rank", "num_votes"], ascending=[True, False]).head(top_n)


def genre_set(value):
    return {item.strip().lower() for item in str(value).split(",") if item.strip()}


SCORING_PROFILES = {
    "balanced": {"semantic": 0.34, "facets": 0.25, "topic": 0.12, "genre": 0.18, "format": 0.05, "quality": 0.06},
    "animation_family": {"semantic": 0.24, "facets": 0.25, "topic": 0.12, "genre": 0.29, "format": 0.06, "quality": 0.04},
    "mood_first": {"semantic": 0.25, "facets": 0.40, "topic": 0.10, "genre": 0.15, "format": 0.05, "quality": 0.05},
    "story_first": {"semantic": 0.50, "facets": 0.20, "topic": 0.12, "genre": 0.10, "format": 0.04, "quality": 0.04},
}


def infer_scoring_profile(seed):
    genres = genre_set(seed["genres"])
    if "animation" in genres and genres & {"adventure", "comedy", "family", "fantasy"}:
        return "animation_family"
    return "balanced"


facet_profile_matrices = {}
for category in FACET_DEFINITIONS:
    raw_scores = facet_vectors[category][1]
    profile = np.exp((raw_scores - raw_scores.max(axis=1, keepdims=True)) / 0.10)
    profile = profile / np.maximum(profile.sum(axis=1, keepdims=True), 1e-12)
    profile = profile / np.maximum(np.linalg.norm(profile, axis=1, keepdims=True), 1e-12)
    facet_profile_matrices[category] = profile


topic_ids = sorted(watch_df["bertopic_topic"].unique())
topic_centroids = {}
for topic_id in topic_ids:
    centroid = embeddings[watch_df["bertopic_topic"].to_numpy() == topic_id].mean(axis=0)
    topic_centroids[topic_id] = centroid / max(np.linalg.norm(centroid), 1e-12)


def recommend_similar_nlp(title, top_n=10, min_votes=0, content_type=None, profile="auto", weights=None):
    matches = find_title(title)
    if matches.empty:
        raise ValueError(f"No match found for title: {title}")

    seed = matches.iloc[0]
    seed_idx = int(seed["row_id"])
    semantic_scores = embeddings @ embeddings[seed_idx]
    semantic_scores[seed_idx] = -1

    results = watch_df.copy()
    results["semantic_similarity"] = semantic_scores
    results["semantic_score"] = np.clip((semantic_scores - 0.10) / 0.50, 0.0, 1.0)

    seed_topic = seed["bertopic_topic"]
    seed_topic_centroid = topic_centroids[seed_topic]
    topic_similarity = results["bertopic_topic"].map(lambda topic_id: float(topic_centroids[topic_id] @ seed_topic_centroid))
    results["topic_similarity"] = topic_similarity
    results["topic_score"] = np.where(
        results["bertopic_topic"] == seed_topic,
        1.0,
        np.clip((topic_similarity - 0.20) / 0.80, 0.0, 1.0),
    )

    seed_genres = genre_set(seed["genres"])
    results["genre_similarity"] = results["genres"].apply(
        lambda value: len(seed_genres & genre_set(value)) / max(len(seed_genres | genre_set(value)), 1)
    )

    facet_scores = []
    for category, profile_matrix in facet_profile_matrices.items():
        facet_scores.append(profile_matrix @ profile_matrix[seed_idx])
    results["facet_similarity"] = np.mean(facet_scores, axis=0)
    results["format_similarity"] = results["content_type"].eq(seed["content_type"]).astype(float)
    results["quality_score"] = results["weighted_rating"].rank(pct=True).fillna(0.5)
    results["topic_reliability"] = np.where(results.get("bertopic_low_confidence", False), 0.92, 1.0)

    selected_profile = infer_scoring_profile(seed) if profile == "auto" else profile
    score_weights = dict(SCORING_PROFILES[selected_profile])
    if weights is not None:
        score_weights.update(weights)
        total_weight = sum(score_weights.values())
        score_weights = {name: value / total_weight for name, value in score_weights.items()}
    results["scoring_profile"] = selected_profile
    results["final_score"] = (
        results["semantic_score"] * score_weights["semantic"]
        + results["facet_similarity"] * score_weights["facets"]
        + results["topic_score"] * score_weights["topic"]
        + results["genre_similarity"] * score_weights["genre"]
        + results["format_similarity"] * score_weights["format"]
        + results["quality_score"] * score_weights["quality"]
    ) * results["topic_reliability"]
    results["match_score"] = (results["final_score"] * 100).round(1)

    if min_votes:
        results = results[results["num_votes"] >= min_votes].copy()
    if content_type is not None:
        results = results[results["content_type"] == content_type].copy()
    results = results[results["row_id"] != seed_idx].copy()

    display_columns = [
        "tconst", "primary_title", "release_year", "content_type", "genres",
        "bertopic_topic", "bertopic_topic_label", "bertopic_low_confidence", "semantic_similarity", "facet_similarity", "topic_similarity",
        "genre_similarity", "scoring_profile", "match_score", "final_score", "description",
    ]
    return results.sort_values("final_score", ascending=False).head(top_n)[display_columns]

recommend_similar_nlp("Parasite", top_n=10, min_votes=30_000)


,tconst,primary_title,release_year,content_type,genres,bertopic_topic,bertopic_topic_label,bertopic_low_confidence,semantic_similarity,facet_similarity,topic_similarity,genre_similarity,scoring_profile,match_score,final_score,description
5284,tt7282468,Burning,2018,movie,"Drama,Mystery,Thriller",13,"East Asian Cinema, Anime, and Serialized Action",False,0.544144,0.823513,1.000000,0.666667,balanced,84.2,0.842468,"Burning (Korean: 버닝) is a 2018 South Korean psychological thriller film co-written, produced, and directed by Lee Chang-dong. The film is based on the short story ""Barn Burning"" from The Elephant Vanishes by Haruki M..."
2210,tt1216496,Mother,2009,movie,"Crime,Drama,Mystery",13,"East Asian Cinema, Anime, and Serialized Action",False,0.594153,0.871949,1.000000,0.250000,balanced,82.0,0.819691,"Mother (Korean: 마더) is a 2009 South Korean neo-noir psychological thriller film directed by Bong Joon Ho, starring Kim Hye-ja and Won Bin. The plot follows a mother who, after her intellectually disabled son is accus..."
661,tt0260991,Joint Security Area,2000,movie,"Action,Drama,Thriller",13,"East Asian Cinema, Anime, and Serialized Action",False,0.468333,0.863648,1.000000,0.666667,balanced,80.5,0.804605,"Joint Security Area is a 2000 South Korean mystery thriller film directed and co-written by Park Chan-wook and based on the novel DMZ by Park Sang-yeon. It is Park Chan-wook's third film as director, and is frequentl..."
2154,tt1190539,The Chaser,2008,movie,"Action,Crime,Drama",13,"East Asian Cinema, Anime, and Serialized Action",False,0.575877,0.798353,1.000000,0.250000,balanced,79.1,0.790654,The Chaser (Korean: 추격자) is a 2008 South Korean action thriller film starring Kim Yoon-seok and Ha Jung-woo. It was directed by Na Hong-jin in his directorial debut. Inspired by real-life Korean serial killer Yoo You...
937,tt0364569,Oldboy,2003,movie,"Action,Drama,Mystery",13,"East Asian Cinema, Anime, and Serialized Action",False,0.524725,0.881217,1.000000,0.250000,balanced,78.3,0.783219,"Oldboy (Korean: 올드보이) is a 2003 South Korean action thriller film directed and co-written by Park Chan-wook. A loose adaptation of the Japanese manga Old Boy by Garon Tsuchiya and Nobuaki Minegishi, the film follows ..."
4690,tt5215952,The Wailing,2016,movie,"Drama,Horror,Mystery",13,"East Asian Cinema, Anime, and Serialized Action",False,0.539163,0.919251,1.000000,0.250000,balanced,78.1,0.780618,"The Wailing is a 2016 South Korean horror film written and directed by Na Hong-jin and starring Kwak Do-won, Hwang Jung-min and Chun Woo-hee. The film centers on a policeman who investigates a series of mysterious ki..."
1296,tt0468492,The Host,2006,movie,"Drama,Horror,Sci-Fi",13,"East Asian Cinema, Anime, and Serialized Action",False,0.623287,0.868499,1.000000,0.250000,balanced,78.0,0.780007,The Host is a 2006 monster film directed and co-written by Bong Joon Ho. It stars Song Kang-ho as food stand vendor Park Gang-du whose daughter Hyun-seo is kidnapped by a creature dwelling around the Han River in Seo...
1137,tt0423866,3-Iron,2004,movie,"Crime,Drama,Romance",13,"East Asian Cinema, Anime, and Serialized Action",False,0.522771,0.858738,1.000000,0.250000,balanced,77.0,0.770357,"Tae-suk (Jae Hee) is a loner who drives around on his motorbike, taping takeout menus over the keyholes of front doors and breaking into apartments where the menus have not been removed. He lives in those apartments ..."
2566,tt1379182,Dogtooth,2009,movie,"Drama,Thriller",3,"Thriller, Crime, and High-Stakes Action",False,0.434407,0.871451,0.714941,1.000000,balanced,76.5,0.764833,"A controlling, manipulative father (Christos Stergioglou) locks his three adult offspring in a state of perpetual childhood by keeping them prisoner within the sprawling family compound. The children are bored to tea..."
944,tt0365737,Syriana,2005,movie,"Drama,Thriller",5,"Horror, Supernatural Threats, and Dark Suspense",False,0.443873,0.881357,0.682440,1.000000,balanced,76.0,0.760407,"Syriana is a 2005 Americ

In [10]:
from difflib import get_close_matches

GENRE_INTENT_LEXICON = {
    "Action": ["action", "exciting", "intense", "fight", "combat", "explosive", "adrenaline", "chase", "martial arts", "battle"],
    "Adventure": ["adventure", "journey", "quest", "exploration", "epic journey", "discovery", "treasure", "mission"],
    "Animation": ["animated", "animation", "anime", "cartoon"],
    "Biography": ["biography", "biopic", "real person", "life story", "based on a person"],
    "Comedy": ["funny", "comedy", "comedic", "hilarious", "laugh", "lighthearted", "witty", "joyous", "joyful", "cheerful", "silly", "goofy"],
    "Crime": ["crime", "criminal", "gangster", "mafia", "heist", "detective", "police", "murder", "law", "court"],
    "Documentary": ["documentary", "nonfiction", "non-fiction", "real life", "true story documentary", "educational"],
    "Drama": ["drama", "character driven", "serious drama"],
    "Family": ["family friendly", "family movie", "wholesome family", "for the family", "kids", "children", "safe", "all ages"],
    "Fantasy": ["fantasy", "magic", "magical", "mythical", "kingdom", "fairy tale", "fairytale", "wizard"],
    "Horror": ["horror", "scary", "terrifying", "creepy", "haunted", "zombie", "monster", "slasher", "demon"],
    "Music": ["music", "musician", "band", "concert", "song", "singer", "artist"],
    "Musical": ["musical", "singing", "song and dance", "songs"],
    "Mystery": ["mystery", "mysterious", "whodunit", "investigation", "puzzle", "mind bending", "mind-bending", "psychological", "twist"],
    "Romance": ["romance", "romantic", "love story", "falling in love", "relationship", "date night"],
    "Sci-Fi": ["sci-fi", "science fiction", "scifi", "futuristic", "future", "space", "alien", "robot", "technology", "cyberpunk", "dystopian"],
    "Sport": ["sport", "sports", "athlete", "football", "basketball", "racing", "competition"],
    "Thriller": ["thriller", "tense", "intense", "suspense", "suspenseful", "dangerous", "edge of my seat", "paranoid"],
    "War": ["war", "battlefield", "soldier", "military", "occupation", "army"],
    "Western": ["western", "cowboy", "frontier", "outlaw"],
}

PROMPT_FACET_HINTS = {
    "mood": {
        "comforting": ["comforting", "cozy", "warm", "wholesome", "feel good", "gentle", "soft", "safe", "calm"],
        "joyful": ["joyful", "joyous", "cheerful", "happy", "bright", "good mood", "delightful"],
        "sad": ["sad", "cry", "tearjerker", "heartbreaking", "heartbroken", "grief", "depressing", "melancholy"],
        "tense": ["tense", "intense", "suspense", "edge of my seat", "anxious", "stressful", "high stakes"],
        "romantic": ["romantic", "romance", "love story", "passionate", "date night"],
        "funny": ["funny", "hilarious", "comedy", "comedic", "laugh", "witty", "silly", "goofy"],
        "dark": ["dark", "disturbing", "grim", "bleak", "violent", "heavy", "traumatic"],
        "hopeful": ["hopeful", "uplifting", "inspiring", "optimistic", "positive"],
        "adventurous": ["exciting", "adventurous", "adventure", "journey", "quest", "exploration"],
        "mind_bending": ["mind bending", "mind-bending", "surreal", "twisty", "reality bending", "confusing in a good way"],
        "inspiring": ["motivating", "inspiring", "inspirational", "ambitious", "underdog", "triumph"],
    },
    "tone": {
        "gentle": ["gentle", "soft", "calm", "soothing", "relaxed", "not intense"],
        "gritty": ["gritty", "raw", "realistic", "street", "harsh"],
        "whimsical": ["whimsical", "charming", "quirky", "playful", "magical"],
        "sincere": ["sincere", "heartfelt", "earnest", "intimate", "honest"],
        "satirical": ["satirical", "satire", "social critique", "ironic", "mocking"],
        "bleak": ["bleak", "hopeless", "oppressive", "despair", "depressing"],
        "suspenseful": ["suspenseful", "suspense", "tense", "thriller", "intense"],
        "epic": ["epic", "grand", "sweeping", "spectacle", "large scale"],
    },
    "themes": {
        "family": ["family", "parent", "children", "siblings", "home"],
        "friendship": ["friendship", "friends", "found family", "companionship", "best friend"],
        "identity": ["identity", "self discovery", "finding myself", "belonging", "who am i"],
        "revenge": ["revenge", "vengeance", "payback"],
        "class_conflict": ["class conflict", "inequality", "rich and poor", "privilege"],
        "survival": ["survival", "survive", "apocalypse", "disaster", "zombie"],
        "grief": ["grief", "mourning", "loss", "bereavement", "heartbreak"],
        "ambition": ["ambition", "success", "fame", "competition", "career"],
        "morality": ["morality", "ethical", "guilt", "moral dilemma", "justice"],
        "love": ["love", "romance", "relationship", "affection", "connection"],
        "healing": ["healing", "recovery", "forgiveness", "second chances", "therapy"],
        "coming_of_age": ["coming of age", "teen", "teenage", "growing up", "school life"],
    },
    "conflict_type": {
        "internal": ["internal struggle", "trauma", "guilt", "personal struggle", "self doubt"],
        "interpersonal": ["relationship conflict", "betrayal", "rivalry", "family conflict"],
        "societal": ["society", "oppression", "injustice", "discrimination", "politics"],
        "criminal": ["crime", "murder", "police", "heist", "gangster", "detective"],
        "survival": ["survival", "escape", "hunted", "trapped", "zombie", "alien invasion"],
        "supernatural": ["supernatural", "ghost", "demon", "magic", "monster", "curse"],
        "competition": ["competition", "contest", "tournament", "audition", "rivalry"],
    },
    "pace": {
        "slow_burn": ["slow burn", "slow-burn", "quiet", "contemplative", "patient"],
        "steady": ["steady", "balanced pace", "not too slow", "medium pace"],
        "fast_paced": ["fast paced", "fast-paced", "exciting", "intense", "urgent", "action packed"],
        "chaotic": ["chaotic", "frantic", "wild", "unpredictable"],
    },
    "emotional_arc": {
        "uplifting": ["uplifting", "feel good", "hopeful", "inspiring", "happy ending", "positive"],
        "tragic": ["tragic", "devastating", "doomed", "heartbreaking", "sad ending"],
        "bittersweet": ["bittersweet", "tender sadness", "mixed emotions", "wistful"],
        "cathartic": ["cathartic", "closure", "emotional release", "relief"],
        "unsettling": ["unsettling", "disturbing", "eerie", "ambiguous", "creepy"],
        "escapist": ["escapist", "distracting", "fun", "entertaining", "light"],
    },
    "setting": {
        "war": ["war", "battlefield", "soldier", "occupation"],
        "school": ["school", "high school", "students", "campus"],
        "workplace": ["workplace", "office", "career", "colleagues"],
        "prison": ["prison", "jail", "inmates", "incarceration"],
        "urban_crime": ["urban crime", "city crime", "gangs", "police"],
        "small_town": ["small town", "rural community", "village"],
        "fantasy_world": ["fantasy world", "magic kingdom", "mythical world"],
        "space_scifi": ["futuristic", "future", "space", "sci-fi", "science fiction", "alien", "robot", "cyberpunk"],
        "music_stage": ["stage", "concert", "band", "singer", "theater", "dance"],
        "home_family": ["home", "family home", "household", "neighborhood"],
    },
}

PROMPT_CONCEPT_LEXICON = {
    "zombie_outbreak": {"prompt": ["zombie", "zombies", "undead", "infected outbreak"], "candidate": ["zombie", "zombies", "undead", "infected", "infection outbreak", "living dead"]},
    "futuristic_scifi": {"prompt": ["futuristic", "future", "cyberpunk", "space", "robot", "alien"], "candidate": ["future", "futuristic", "science fiction", "sci-fi", "cyberpunk", "space", "robot", "alien", "artificial intelligence"]},
    "mind_bending": {"prompt": ["mind bending", "mind-bending", "reality bending", "twisty", "surreal"], "candidate": ["time loop", "alternate reality", "dream", "memory", "identity", "surreal", "reality", "nonlinear", "psychological", "twist"]},
    "musical_performance": {"prompt": ["musical", "singing", "song and dance", "music"], "candidate": ["musical", "songs", "singing", "singer", "music", "dance", "composer", "band"]},
    "wholesome_family": {"prompt": ["wholesome", "family movie", "family friendly", "comforting"], "candidate": ["family", "friendship", "heartwarming", "wholesome", "community", "children", "healing", "belonging"]},
    "romantic_love": {"prompt": ["romantic", "romance", "love story", "falling in love"], "candidate": ["romance", "romantic", "love", "relationship", "couple", "marriage"]},
    "crime_investigation": {"prompt": ["detective", "investigation", "murder mystery", "police"], "candidate": ["detective", "investigation", "murder", "police", "case", "crime", "mystery"]},
    "sports_competition": {"prompt": ["sports", "sport", "athlete", "competition", "tournament"], "candidate": ["sport", "team", "athlete", "competition", "coach", "championship", "tournament"]},
}

facet_label_embeddings = {}
for category, labels in FACET_DEFINITIONS.items():
    facet_label_embeddings[category] = embedding_model.encode(
        list(labels.values()),
        batch_size=BATCH_SIZE,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

TYPO_ALIASES = {
    "commedy": "comedy", "comedie": "comedy", "comdy": "comedy", "comdic": "comedic", "funy": "funny",
    "joyus": "joyous", "joyfull": "joyful", "hartbroken": "heartbroken", "heartbrokn": "heartbroken",
    "hartbreak": "heartbreak", "romence": "romance", "romanticc": "romantic", "scarrry": "scary",
    "scarry": "scary", "thriler": "thriller", "mysstery": "mystery", "futurstic": "futuristic",
    "advanture": "adventure", "adventerous": "adventurous", "wholsome": "wholesome", "uplifing": "uplifting",
}
PHRASE_ALIASES = {
    "heart broken": "heartbroken",
    "mindbending": "mind bending",
    "sci fi": "sci-fi",
    "sciencefiction": "science fiction",
    "feelgood": "feel good",
    "light hearted": "lighthearted",
    "i wanna": "i want to",
    "wanna watch": "want to watch",
    "dont want": "do not want",
    "don't want": "do not want",
    "im ": "i am ",
}
NEGATION_CUES = {"not", "no", "never", "without", "avoid", "exclude", "except", "nothing", "dont", "don't"}
NEGATION_PHRASES = ["do not want", "dont want", "don't want", "not looking for", "no more", "nothing", "without", "avoid", "exclude", "not too", "less"]
DESIRED_EXPERIENCE_MARKERS = [
    "i want to watch", "i want something", "i want a", "i want", "want to watch", "want something", "want a", "want",
    "i need something", "i need a", "i need", "need something", "need", "i would like", "i'd like", "looking for",
    "give me", "recommend", "show me", "in the mood for", "watch something", "watch a", "can you suggest",
]
CURRENT_STATE_MARKERS = ["i feel", "i am feeling", "i'm feeling", "im feeling", "feeling", "i am", "i'm", "im", "me feel", "me sad"]
DISTRESS_STATE_HINTS = ["sad", "heartbroken", "heartbreak", "depressed", "down", "lonely", "upset", "tired", "stressed", "anxious", "bad", "awful", "miserable", "crying", "hurt"]
RELIEF_INTENT_TEXT = "desired viewing experience comforting hopeful uplifting funny comedic joyful cheerful lighthearted feel good warm gentle wholesome healing escapist"
SUPPRESSED_RELIEF_FACETS = {"mood": {"sad", "dark"}, "tone": {"bleak", "gritty"}, "themes": {"grief", "revenge"}, "emotional_arc": {"tragic", "unsettling"}}

KNOWN_PROMPT_TERMS = sorted({
    word
    for source in [GENRE_INTENT_LEXICON, PROMPT_FACET_HINTS]
    for values in source.values()
    for items in (values.values() if isinstance(values, dict) else [values])
    for phrase in items
    for word in str(phrase).replace("-", " ").split()
    if len(word) >= 5
} | set(TYPO_ALIASES.values()) | {"heartbroken", "joyous", "comedic", "wholesome", "uplifting"})


def normalize_prompt_language(value):
    text = normalize_title(value).replace("’", "'")
    text = re.sub(r"[^a-z0-9#+' -]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    for old, new in PHRASE_ALIASES.items():
        text = re.sub(r"(?<!\w)" + re.escape(old) + r"(?!\w)", new, text)
    corrected = []
    for token in text.split():
        clean = token.strip(" '")
        if clean in TYPO_ALIASES:
            corrected.append(TYPO_ALIASES[clean])
        elif len(clean) >= 6:
            match = get_close_matches(clean, KNOWN_PROMPT_TERMS, n=1, cutoff=0.89)
            corrected.append(match[0] if match else clean)
        else:
            corrected.append(clean)
    return re.sub(r"\s+", " ", " ".join(corrected)).strip()


def prompt_contains(text, phrase):
    return re.search(r"(?<!\w)" + re.escape(phrase) + r"(?!\w)", text) is not None


def phrase_is_negated(text, phrase, window=4):
    match = re.search(r"(?<!\w)" + re.escape(phrase) + r"(?!\w)", text)
    if not match:
        return False
    before_text = text[:match.start()]
    # Only inspect the current clause. This prevents "not tragic heartbreak, something sincere"
    # from incorrectly negating "sincere" after the comma.
    clause = re.split(r"[,.;]|\bbut\b|\bhowever\b|\bthough\b|\bsomething\b|\bmore\b|\bjust\b", before_text)[-1]
    before = clause.split()[-window:]
    compact_clause = " ".join(before)
    return bool(NEGATION_CUES.intersection(before)) or any(cue in compact_clause for cue in NEGATION_PHRASES)


def first_marker_position(text, markers):
    positions = [text.find(marker) for marker in markers if marker in text]
    return min(positions) if positions else -1


def extract_prompt_exclusions(text):
    excluded_genres = set()
    excluded_facets = defaultdict(set)
    for genre, phrases in GENRE_INTENT_LEXICON.items():
        if any(phrase_is_negated(text, phrase) for phrase in phrases):
            excluded_genres.add(genre)
    for category, labels in PROMPT_FACET_HINTS.items():
        for label, phrases in labels.items():
            if any(phrase_is_negated(text, phrase) for phrase in phrases):
                excluded_facets[category].add(label)
    if any(phrase_is_negated(text, phrase) for phrase in ["scary", "horror", "creepy", "terrifying"]):
        excluded_genres.add("Horror")
        excluded_facets["mood"].update({"dark", "tense"})
        excluded_facets["tone"].update({"suspenseful", "bleak"})
        excluded_facets["emotional_arc"].add("unsettling")
    if any(phrase_is_negated(text, phrase) for phrase in ["sad", "heartbreaking", "heartbreak", "depressing", "tragic", "devastating"]):
        excluded_facets["mood"].add("sad")
        excluded_facets["themes"].add("grief")
        excluded_facets["emotional_arc"].add("tragic")
    if any(phrase in text for phrase in ["do not destroy me", "dont destroy me", "don't destroy me", "not destroy me", "do not wreck me", "not wreck me"]):
        excluded_facets["mood"].update({"sad", "dark"})
        excluded_facets["tone"].update({"bleak", "gritty"})
        excluded_facets["themes"].add("grief")
        excluded_facets["emotional_arc"].update({"tragic", "unsettling"})
    if any(phrase_is_negated(text, phrase) for phrase in ["intense", "tense", "stressful"]):
        excluded_genres.add("Thriller")
        excluded_facets["mood"].add("tense")
        excluded_facets["pace"].add("fast_paced")
    return excluded_genres, dict(excluded_facets)


def parse_prompt_intent(prompt):
    normalized = normalize_prompt_language(prompt)
    desired_pos = first_marker_position(normalized, DESIRED_EXPERIENCE_MARKERS)
    state_pos = first_marker_position(normalized, CURRENT_STATE_MARKERS)
    has_distress_state = any(prompt_contains(normalized, hint) for hint in DISTRESS_STATE_HINTS)
    excluded_genres, excluded_facets = extract_prompt_exclusions(normalized)

    if desired_pos >= 0:
        desired_text = normalized[desired_pos:]
        state_text = normalized[:desired_pos].strip()
        mode = "explicit_desire"
    elif state_pos >= 0 and has_distress_state:
        desired_text = f"{normalized}. {RELIEF_INTENT_TEXT}"
        state_text = normalized
        mode = "relief_from_current_state"
        for category, labels in SUPPRESSED_RELIEF_FACETS.items():
            excluded_facets.setdefault(category, set()).update(labels)
    else:
        desired_text = normalized
        state_text = ""
        mode = "direct_request"

    return {
        "original_prompt": prompt,
        "normalized_prompt": normalized,
        "desired_text": desired_text,
        "state_text": state_text,
        "mode": mode,
        "excluded_genres": excluded_genres,
        "excluded_facets": excluded_facets,
    }


def infer_prompt_genres(prompt, preferred_genres=None, prompt_intent=None):
    prompt_intent = prompt_intent or parse_prompt_intent(prompt)
    if preferred_genres:
        return {genre: 1.0 for genre in preferred_genres if genre not in prompt_intent["excluded_genres"]}
    text = prompt_intent["desired_text"]
    inferred = {}
    for genre, phrases in GENRE_INTENT_LEXICON.items():
        if genre in prompt_intent["excluded_genres"]:
            continue
        hits = sum(prompt_contains(text, phrase) and not phrase_is_negated(text, phrase) for phrase in phrases)
        if hits:
            inferred[genre] = min(1.0, 0.55 + 0.20 * hits)
    if prompt_intent["mode"] == "relief_from_current_state":
        inferred["Comedy"] = max(inferred.get("Comedy", 0.0), 0.85)
        inferred["Family"] = max(inferred.get("Family", 0.0), 0.65)
    return dict(sorted(inferred.items(), key=lambda item: item[1], reverse=True)[:5])


def infer_prompt_concepts(prompt, prompt_intent=None):
    prompt_intent = prompt_intent or parse_prompt_intent(prompt)
    text = prompt_intent["desired_text"]
    return {
        concept: definition["candidate"]
        for concept, definition in PROMPT_CONCEPT_LEXICON.items()
        if any(prompt_contains(text, phrase) and not phrase_is_negated(text, phrase) for phrase in definition["prompt"])
    }


def score_prompt_against_facets(prompt, prompt_embedding=None, prompt_intent=None):
    prompt_facet_scores = {}
    prompt_intent = prompt_intent or parse_prompt_intent(prompt)
    text = prompt_intent["desired_text"]
    excluded_facets = prompt_intent.get("excluded_facets", {})
    if prompt_embedding is None:
        prompt_embedding = embedding_model.encode([text], show_progress_bar=False, convert_to_numpy=True, normalize_embeddings=True)[0]
    for category, labels in FACET_DEFINITIONS.items():
        label_names = list(labels.keys())
        scores = prompt_embedding @ facet_label_embeddings[category].T
        lexical_hits = np.zeros(len(label_names), dtype=float)
        for label_idx, label in enumerate(label_names):
            phrases = PROMPT_FACET_HINTS.get(category, {}).get(label, [])
            lexical_hits[label_idx] = sum(prompt_contains(text, phrase) and not phrase_is_negated(text, phrase) for phrase in phrases)
            if label in excluded_facets.get(category, set()):
                lexical_hits[label_idx] = 0.0
                scores[label_idx] -= 0.28
        boosted_scores = scores + np.minimum(lexical_hits, 2.0) * 0.25
        profile = np.exp((boosted_scores - boosted_scores.max()) / 0.085)
        profile = profile / max(profile.sum(), 1e-12)
        profile = profile / max(np.linalg.norm(profile), 1e-12)
        semantic_relevance = float(np.clip((scores.max() - 0.18) / 0.22, 0.0, 1.0))
        category_weight = float(min(3.0, lexical_hits.sum() * 1.5) + 0.20 * semantic_relevance)
        if prompt_intent["mode"] == "relief_from_current_state" and category in {"mood", "tone", "emotional_arc"}:
            category_weight = max(category_weight, 1.8)
        prompt_facet_scores[category] = {"label_names": label_names, "scores": scores, "profile": profile, "lexical_hits": lexical_hits, "category_weight": category_weight}
    return prompt_facet_scores


def build_prompt_intent(prompt, inferred_genres, prompt_facet_scores, prompt_intent=None):
    prompt_intent = prompt_intent or parse_prompt_intent(prompt)
    parts = ["Desired viewing experience: " + prompt_intent["desired_text"]]
    if prompt_intent["state_text"]:
        parts.append("Current feeling is context, not necessarily desired content: " + prompt_intent["state_text"])
    if inferred_genres:
        parts.append("Preferred genres and story style: " + ", ".join(inferred_genres))
    if prompt_intent["excluded_genres"]:
        parts.append("Avoid genres: " + ", ".join(sorted(prompt_intent["excluded_genres"])))
    for category, facet_data in prompt_facet_scores.items():
        if facet_data["lexical_hits"].sum() <= 0 and prompt_intent["mode"] != "relief_from_current_state":
            continue
        top_idx = int(np.argmax(facet_data["lexical_hits"] + facet_data["scores"]))
        label = facet_data["label_names"][top_idx]
        parts.append(f"{category}: {label}. {FACET_DEFINITIONS[category][label]}")
    return " ".join(parts)


### Natural-Language Prompt Intent, Ranking, And Score Calibration

In [11]:
def candidate_text_for_prompt_matching(results):
    parts = [
        results["primary_title"].fillna(""),
        results["genres"].fillna(""),
        results.get("bertopic_topic_label", pd.Series("", index=results.index)).fillna(""),
        results["description"].fillna(""),
    ]
    return (parts[0] + " " + parts[1] + " " + parts[2] + " " + parts[3]).str.casefold()


def genre_membership_score(genres_value, weighted_genres):
    if not weighted_genres:
        return 0.5
    row_genres = genre_set(genres_value)
    total_weight = sum(weighted_genres.values())
    return sum(weight for genre, weight in weighted_genres.items() if genre.casefold() in row_genres) / max(total_weight, 1e-12)


def excluded_genre_score(genres_value, excluded_genres):
    if not excluded_genres:
        return 0.0
    row_genres = genre_set(genres_value)
    return float(any(genre.casefold() in row_genres for genre in excluded_genres))


def excluded_facet_signal(prompt_intent, results):
    signals = []
    for category, labels in prompt_intent.get("excluded_facets", {}).items():
        if not labels or category not in facet_profile_matrices:
            continue
        label_names = facet_vectors[category][0]
        label_indices = [label_names.index(label) for label in labels if label in label_names]
        if not label_indices:
            continue
        raw = facet_profile_matrices[category][:, label_indices].max(axis=1)
        signals.append(pd.Series(raw, index=watch_df.index).reindex(results.index).fillna(0.0).to_numpy())
    if not signals:
        return np.zeros(len(results), dtype=float)
    return np.max(np.vstack(signals), axis=0)


def recommend_from_prompt_nlp(
    prompt,
    top_n=10,
    min_votes=0,
    content_type=None,
    preferred_genres=None,
    profile="prompt_intent_v2",
    weights=None,
    min_match_score=75.0,
    return_diagnostics=True,
):
    prompt_intent = parse_prompt_intent(prompt)
    raw_prompt_embedding = embedding_model.encode(
        [prompt_intent["desired_text"]], show_progress_bar=False, convert_to_numpy=True, normalize_embeddings=True
    )[0]
    inferred_genres = infer_prompt_genres(prompt, preferred_genres, prompt_intent)
    inferred_concepts = infer_prompt_concepts(prompt, prompt_intent)
    prompt_facet_scores = score_prompt_against_facets(prompt, raw_prompt_embedding, prompt_intent)
    intent_text = build_prompt_intent(prompt, inferred_genres, prompt_facet_scores, prompt_intent)
    intent_embedding = embedding_model.encode(
        [intent_text], show_progress_bar=False, convert_to_numpy=True, normalize_embeddings=True
    )[0]
    query_embedding = raw_prompt_embedding * 0.35 + intent_embedding * 0.65
    query_embedding = query_embedding / max(np.linalg.norm(query_embedding), 1e-12)

    results = watch_df.copy()
    raw_semantic = embeddings @ raw_prompt_embedding
    intent_semantic = embeddings @ intent_embedding
    results["semantic_similarity"] = np.maximum(raw_semantic, intent_semantic * 0.80 + raw_semantic * 0.20)

    topic_id_array = np.asarray(topic_ids)
    topic_centroid_matrix = np.vstack([topic_centroids[topic_id] for topic_id in topic_ids])
    prompt_topic_scores = topic_centroid_matrix @ query_embedding
    prompt_topic = int(topic_id_array[int(np.argmax(prompt_topic_scores))])
    topic_score_map = dict(zip(topic_ids, prompt_topic_scores))
    results["prompt_topic"] = prompt_topic
    results["topic_similarity"] = results["bertopic_topic"].map(topic_score_map).astype(float)

    weighted_facet_scores = []
    facet_category_weights = []
    for category, profile_matrix in facet_profile_matrices.items():
        facet_data = prompt_facet_scores[category]
        category_weight = facet_data["category_weight"]
        if category_weight <= 0:
            continue
        weighted_facet_scores.append((profile_matrix @ facet_data["profile"]) * category_weight)
        facet_category_weights.append(category_weight)
    if weighted_facet_scores:
        results["facet_similarity"] = np.sum(weighted_facet_scores, axis=0) / sum(facet_category_weights)
    else:
        results["facet_similarity"] = 0.5

    results["genre_similarity"] = results["genres"].map(lambda value: genre_membership_score(value, inferred_genres))

    candidate_text = candidate_text_for_prompt_matching(results)
    if inferred_concepts:
        concept_matches = []
        for concept_phrases in inferred_concepts.values():
            concept_matches.append(candidate_text.map(
                lambda text: min(1.0, sum(prompt_contains(text, phrase) for phrase in concept_phrases) / 2.0)
            ).to_numpy())
        results["concept_similarity"] = np.mean(concept_matches, axis=0)
    else:
        results["concept_similarity"] = 0.5

    results["excluded_genre_signal"] = results["genres"].map(
        lambda value: excluded_genre_score(value, prompt_intent["excluded_genres"])
    )
    results["excluded_facet_signal"] = excluded_facet_signal(prompt_intent, results)
    results["negative_match_signal"] = np.maximum(results["excluded_genre_signal"], results["excluded_facet_signal"])
    results["genre_miss_signal"] = np.where(inferred_genres, 1.0 - results["genre_similarity"], 0.0)

    if content_type is not None:
        results = results[results["content_type"] == content_type].copy()
    if min_votes:
        results = results[results["num_votes"] >= min_votes].copy()
    if results.empty:
        return results

    results["semantic_rank"] = results["semantic_similarity"].rank(pct=True)
    results["facet_rank"] = results["facet_similarity"].rank(pct=True)
    results["topic_rank"] = results["topic_similarity"].rank(pct=True)
    results["quality_score"] = results["weighted_rating"].rank(pct=True).fillna(0.5)
    results["concept_rank"] = results["concept_similarity"].rank(pct=True)
    results["semantic_absolute"] = np.clip((results["semantic_similarity"] - 0.12) / 0.38, 0.0, 1.0)
    results["facet_absolute"] = np.clip((results["facet_similarity"] - 0.45) / 0.50, 0.0, 1.0)
    results["topic_absolute"] = np.clip((results["topic_similarity"] - 0.05) / 0.45, 0.0, 1.0)
    results["topic_reliability"] = np.where(results.get("bertopic_low_confidence", False), 0.94, 1.0)

    score_weights = {"semantic": 0.30, "facets": 0.25, "genre": 0.20, "concept": 0.15, "topic": 0.04, "quality": 0.06}
    if not inferred_genres:
        score_weights = {"semantic": 0.40, "facets": 0.31, "genre": 0.0, "concept": 0.16, "topic": 0.06, "quality": 0.07}
    if not inferred_concepts:
        concept_weight = score_weights.pop("concept")
        score_weights["semantic"] += concept_weight * 0.50
        score_weights["facets"] += concept_weight * 0.50
    if prompt_intent["excluded_genres"] or prompt_intent["excluded_facets"]:
        score_weights["facets"] += 0.03
        score_weights["semantic"] -= 0.02
        score_weights["quality"] -= 0.01
    if weights is not None:
        score_weights.update(weights)
    total_weight = sum(score_weights.values())
    score_weights = {name: value / total_weight for name, value in score_weights.items()}

    base_score = (
        results["semantic_rank"] * score_weights["semantic"]
        + results["facet_rank"] * score_weights["facets"]
        + results["genre_similarity"] * score_weights.get("genre", 0.0)
        + results["concept_rank"] * score_weights.get("concept", 0.0)
        + results["topic_rank"] * score_weights["topic"]
        + results["quality_score"] * score_weights["quality"]
    )
    results["final_score"] = np.clip(
        (base_score - results["negative_match_signal"] * 0.30 - results["genre_miss_signal"] * 0.10)
        * results["topic_reliability"],
        0.0,
        1.0,
    )
    results["absolute_fit"] = np.clip((
        results["semantic_absolute"] * score_weights["semantic"]
        + results["facet_absolute"] * score_weights["facets"]
        + results["genre_similarity"] * score_weights.get("genre", 0.0)
        + results["concept_similarity"] * score_weights.get("concept", 0.0)
        + results["topic_absolute"] * score_weights["topic"]
        + results["quality_score"] * score_weights["quality"]
        - results["negative_match_signal"] * 0.30
        - results["genre_miss_signal"] * 0.10
    ), 0.0, 1.0)
    results["relative_confidence"] = results["final_score"].rank(pct=True)
    results["match_score"] = ((results["absolute_fit"] * 0.40 + results["relative_confidence"] * 0.60) * 100).clip(0, 100).round(1)
    results["scoring_profile"] = profile
    results["prompt_intent_mode"] = prompt_intent["mode"]
    results["normalized_prompt"] = prompt_intent["normalized_prompt"]
    results["interpreted_request"] = prompt_intent["desired_text"]
    results["inferred_genres"] = ", ".join(inferred_genres) if inferred_genres else "none"
    results["excluded_genres"] = ", ".join(sorted(prompt_intent["excluded_genres"])) if prompt_intent["excluded_genres"] else "none"
    excluded_facet_parts = []
    for category, labels in prompt_intent.get("excluded_facets", {}).items():
        if labels:
            excluded_facet_parts.append(f"{category}:" + "/".join(sorted(labels)))
    results["excluded_facets"] = "; ".join(excluded_facet_parts) if excluded_facet_parts else "none"
    results["inferred_concepts"] = ", ".join(inferred_concepts) if inferred_concepts else "none"
    results["prompt_topic_label"] = topic_label_map.get(prompt_topic, "unknown") if "topic_label_map" in globals() else str(prompt_topic)

    ranked = results.sort_values(["final_score", "match_score"], ascending=False)
    if min_match_score is not None:
        filtered = ranked[ranked["match_score"] >= min_match_score]
        if len(filtered) >= min(top_n, 3):
            ranked = filtered

    display_columns = [
        "tconst", "primary_title", "release_year", "content_type", "genres",
        "bertopic_topic_label", "prompt_intent_mode", "normalized_prompt", "interpreted_request",
        "inferred_genres", "excluded_genres", "excluded_facets", "inferred_concepts", "negative_match_signal", "genre_miss_signal",
        "semantic_similarity", "facet_similarity", "topic_similarity", "genre_similarity", "concept_similarity",
        "match_score", "final_score", "description",
    ]
    if not return_diagnostics:
        display_columns = [
            "primary_title", "release_year", "content_type", "genres", "match_score", "description"
        ]
    display_columns = [column for column in display_columns if column in ranked.columns]
    return ranked.head(top_n)[display_columns]

PROMPT_EVALUATION_CASES = [
    {"category": "clear genre mood", "prompt": "I want something exciting, futuristic, and intense"},
    {"category": "clear comfort", "prompt": "I want a comforting, funny, wholesome family movie"},
    {"category": "dark specific", "prompt": "Give me a dark psychological and mind-bending thriller"},
    {"category": "romantic mixed emotion", "prompt": "I want a romantic, bittersweet musical with strong emotions"},
    {"category": "specific horror", "prompt": "I want a tense zombie survival horror movie"},
    {"category": "current feeling only", "prompt": "I feel sad and heartbroken"},
    {"category": "current feeling plus desire", "prompt": "I feel sad and heartbroken and I want to watch something joyous and comedic"},
    {"category": "negation", "prompt": "I want something funny but not scary"},
    {"category": "typos negation", "prompt": "I dont want sad movie, need joyful commedy"},
    {"category": "broken english", "prompt": "me sad heart broken want watch happy funny"},
    {"category": "multi exclusion comfort", "prompt": "I am exhausted and anxious, no horror, no depressing stuff, just something warm and easy to watch"},
    {"category": "soft sci-fi", "prompt": "I want sci fi but not too intense, more hopeful and emotional than violent"},
    {"category": "clever mystery exclusion", "prompt": "Give me a clever mystery with twists but nothing violent or disturbing"},
    {"category": "inspiring true story", "prompt": "I want an inspiring true story or documentary, but not war and not too bleak"},
    {"category": "romance exclusion", "prompt": "I want romance, but not tragic heartbreak, something sincere and uplifting"},
    {"category": "bad grammar comedy", "prompt": "my english bad but i want movie make me laugh no sad no scary"},
    {"category": "anger regulation", "prompt": "I feel angry and stressed, I need something calm, funny, and not dark"},
    {"category": "animation nuance", "prompt": "I want animated or anime action that is fun and adventurous, not horror"},
    {"category": "work fatigue", "prompt": "After a long work day I want light smart comedy, no crime, no heavy drama"},
    {"category": "vague but usable", "prompt": "Something beautiful, human, a little emotional, but please do not destroy me"},
]


def score_band(score):
    if pd.isna(score):
        return "no_results"
    if score >= 85:
        return "strong"
    if score >= 75:
        return "usable"
    if score >= 65:
        return "borderline"
    return "weak"


def prompt_result_warning(prompt_results):
    if prompt_results.empty:
        return "no results passed the score filter"
    top_score = prompt_results["match_score"].max()
    warnings = []
    if top_score < 75:
        warnings.append("top score below usable threshold")
    if prompt_results["negative_match_signal"].head(3).max() > 0.55:
        warnings.append("top results still contain excluded mood/genre signals")
    if prompt_results["genre_miss_signal"].head(3).mean() > 0.55:
        warnings.append("top results weakly match inferred genre intent")
    if len(prompt_results) < 3:
        warnings.append("fewer than three high-confidence results")
    return "; ".join(warnings) if warnings else "ok"


prompt_evaluation_rows = []
prompt_summary_rows = []
for case in PROMPT_EVALUATION_CASES:
    evaluation_prompt = case["prompt"]
    print("\n" + "=" * 100)
    print("CATEGORY:", case["category"])
    print("PROMPT:", evaluation_prompt)
    prompt_results = recommend_from_prompt_nlp(
        evaluation_prompt, top_n=10, min_votes=30_000, min_match_score=70.0
    )
    top_score = prompt_results["match_score"].max() if len(prompt_results) else np.nan
    print("Returned:", len(prompt_results), "Top score:", top_score, "Band:", score_band(top_score), "Warning:", prompt_result_warning(prompt_results))
    display(prompt_results)
    prompt_summary_rows.append({
        "category": case["category"],
        "prompt": evaluation_prompt,
        "result_count": len(prompt_results),
        "top_match_score": top_score,
        "score_band": score_band(top_score),
        "warning": prompt_result_warning(prompt_results),
        "top_title": prompt_results.iloc[0]["primary_title"] if len(prompt_results) else "none",
        "top_genres": prompt_results.iloc[0]["genres"] if len(prompt_results) else "none",
        "inferred_genres": prompt_results.iloc[0]["inferred_genres"] if len(prompt_results) else "none",
        "excluded_genres": prompt_results.iloc[0]["excluded_genres"] if len(prompt_results) else "none",
        "excluded_facets": prompt_results.iloc[0]["excluded_facets"] if len(prompt_results) else "none",
    })
    for rank, row in enumerate(prompt_results.itertuples(index=False), start=1):
        prompt_evaluation_rows.append({
            "category": case["category"],
            "prompt": evaluation_prompt,
            "rank": rank,
            "primary_title": row.primary_title,
            "genres": row.genres,
            "match_score": row.match_score,
            "final_score": row.final_score,
            "intent_mode": row.prompt_intent_mode,
            "inferred_genres": row.inferred_genres,
            "excluded_genres": row.excluded_genres,
            "excluded_facets": row.excluded_facets,
            "negative_match_signal": row.negative_match_signal,
            "genre_miss_signal": row.genre_miss_signal,
        })

prompt_evaluation = pd.DataFrame(prompt_evaluation_rows)
prompt_summary = pd.DataFrame(prompt_summary_rows)
prompt_evaluation.to_csv(PROMPT_EVALUATION_PATH, index=False)
prompt_summary.to_csv(DATA_DIR / "prompt_recommendation_stress_summary.csv", index=False)
print("Saved prompt evaluation:", PROMPT_EVALUATION_PATH)
print("Saved prompt stress summary:", DATA_DIR / "prompt_recommendation_stress_summary.csv")
display(prompt_summary)



CATEGORY: clear genre mood
PROMPT: I want something exciting, futuristic, and intense


Returned: 10 Top score: 90.7 Band: strong Warning: ok


,tconst,primary_title,release_year,content_type,genres,bertopic_topic_label,prompt_intent_mode,normalized_prompt,interpreted_request,inferred_genres,excluded_genres,excluded_facets,inferred_concepts,negative_match_signal,genre_miss_signal,semantic_similarity,facet_similarity,topic_similarity,genre_similarity,concept_similarity,match_score,final_score,description
5131,tt6723592,Tenet,2020,movie,"Action,Sci-Fi,Thriller","Horror, Supernatural Threats, and Dark Suspense",explicit_desire,i want something exciting futuristic and intense,i want something exciting futuristic and intense,"Action, Sci-Fi, Thriller",none,none,futuristic_scifi,0.0,0.000000,0.435776,0.654229,0.515422,1.000000,1.0,90.7,0.952635,"Tenet is a 2020 science fiction action thriller film written and directed by Christopher Nolan, who also produced it with his wife Emma Thomas. It stars John David Washington, Robert Pattinson, Elizabeth Debicki, Dim..."
3402,tt1954347,Continuum,2012,tvSeries,"Action,Sci-Fi,Thriller","Science Fiction, Technology, and Speculative Worlds",explicit_desire,i want something exciting futuristic and intense,i want something exciting futuristic and intense,"Action, Sci-Fi, Thriller",none,none,futuristic_scifi,0.0,0.000000,0.419351,0.441775,0.508373,1.000000,1.0,86.5,0.904081,"Continuum is a Canadian science fiction television series created by Simon Barry that premiered on Showcase on May 27, 2012, and ran for four seasons. It was produced by Reunion Pictures, Boy Meets Girl Film Company,..."
3071,tt1631867,Edge of Tomorrow,2014,movie,"Action,Adventure,Sci-Fi","Horror, Supernatural Threats, and Dark Suspense",explicit_desire,i want something exciting futuristic and intense,i want something exciting futuristic and intense,"Action, Sci-Fi, Thriller",none,none,futuristic_scifi,0.0,0.306122,0.378403,0.670958,0.515422,0.693878,1.0,86.5,0.882617,"Edge of Tomorrow is a 2014 American science fiction action film directed by Doug Liman and written by Christopher McQuarrie, Jez Butterworth and John-Henry Butterworth, based on the Japanese light novel All You Need ..."
2239,tt12327578,Star Trek: Strange New Worlds,2022,tvSeries,"Action,Adventure,Sci-Fi","Science Fiction, Technology, and Speculative Worlds",explicit_desire,i want something exciting futuristic and intense,i want something exciting futuristic and intense,"Action, Sci-Fi, Thriller",none,none,futuristic_scifi,0.0,0.306122,0.376868,0.696938,0.508373,0.693878,1.0,86.7,0.875510,"Star Trek: Strange New Worlds is an American science fiction television series created by Akiva Goldsman, Alex Kurtzman, and Jenny Lumet for the streaming service Paramount+. It is the 11th Star Trek series and debut..."
101,tt1375666,Inception,2010,movie,"Action,Adventure,Sci-Fi","Horror, Supernatural Threats, and Dark Suspense",explicit_desire,i want something exciting futuristic and intense,i want something exciting futuristic and intense,"Action, Sci-Fi, Thriller",none,none,futuristic_scifi,0.0,0.306122,0.333875,0.598837,0.515422,0.693878,1.0,84.0,0.868077,"Inception is a 2010 science fiction action film written and directed by Christopher Nolan, who also produced it with Emma Thomas, his wife. The film stars Leonardo DiCaprio as a professional thief who steals informat..."
3766,tt2397535,Predestination,2014,movie,"Action,Drama,Sci-Fi","Horror, Supernatural Threats, and Dark Suspense",explicit_desire,i want something exciting futuristic and intense,i want something exciting futuristic and intense,"Action, Sci-Fi, Thriller",none,none,futuristic_scifi,0.0,0.306122,0.386652,0.678481,0.515422,0.693878,1.0,86.1,0.865196,"Predestination is a 2014 Australian science fiction thriller film written and directed by Michael and Peter Spierig. The film stars Ethan Hawke, Sarah Snook, and Noah Taylor, and is based on the 1959 short story "" '—..."
472,tt0133093,The Matrix,1999,movie,"Action,Sci-Fi","Horror, Supernatural Threats, and Dark Suspense",explicit_desire,i want something exciting futuristic and intense,i want something 


CATEGORY: clear comfort
PROMPT: I want a comforting, funny, wholesome family movie


Returned: 10 Top score: 85.4 Band: strong Warning: ok


...


CATEGORY: vague but usable
PROMPT: Something beautiful, human, a little emotional, but please do not destroy me


Returned: 10 Top score: 76.5 Band: usable Warning: ok


,tconst,primary_title,release_year,content_type,genres,bertopic_topic_label,prompt_intent_mode,normalized_prompt,interpreted_request,inferred_genres,excluded_genres,excluded_facets,inferred_concepts,negative_match_signal,genre_miss_signal,semantic_similarity,facet_similarity,topic_similarity,genre_similarity,concept_similarity,match_score,final_score,description
1269,tt0461770,Enchanted,2007,movie,"Adventure,Animation,Comedy","Holiday, Family, and Comfort Viewing",direct_request,something beautiful human a little emotional but please do not destroy me,something beautiful human a little emotional but please do not destroy me,none,none,mood:dark/sad; tone:bleak/gritty; themes:grief; emotional_arc:tragic/unsettling,none,0.209633,0.0,0.336436,0.650878,0.248366,0.5,0.5,76.5,0.846855,"Enchanted is a 2007 American live-action animated musical fantasy romantic comedy film directed by Kevin Lima and written by Bill Kelly. Co-produced by Walt Disney Pictures, Josephson Entertainment, and Right Coast P..."
3175,tt1714206,The Spectacular Now,2013,movie,"Drama,Romance","Romance, Relationships, and Romantic Comedy",direct_request,something beautiful human a little emotional but please do not destroy me,something beautiful human a little emotional but please do not destroy me,none,none,mood:dark/sad; tone:bleak/gritty; themes:grief; emotional_arc:tragic/unsettling,none,0.311281,0.0,0.253239,0.747353,0.349560,0.5,0.5,75.0,0.844804,"The Spectacular Now is a 2013 American coming-of-age romantic drama film directed by James Ponsoldt, from a screenplay written by Scott Neustadter and Michael H. Weber, based on the 2008 novel of the same name by Tim..."
505,tt0169547,American Beauty,1999,movie,Drama,"Romance, Relationships, and Romantic Comedy",direct_request,something beautiful human a little emotional but please do not destroy me,something beautiful human a little emotional but please do not destroy me,none,none,mood:dark/sad; tone:bleak/gritty; themes:grief; emotional_arc:tragic/unsettling,none,0.489574,0.0,0.309601,0.713117,0.349560,0.5,0.5,76.0,0.836170,"Lester Burnham, a middle-aged media executive in suburbia, despises his job and is unhappily married to neurotic, status-obsessed Carolyn. Their 16-year-old daughter and only child, Jane, resents her parents and has ..."
3832,tt2555736,The Second Best Exotic Marigold Hotel,2015,movie,"Comedy,Drama,Romance","Romance, Relationships, and Romantic Comedy",direct_request,something beautiful human a little emotional but please do not destroy me,something beautiful human a little emotional but please do not destroy me,none,none,mood:dark/sad; tone:bleak/gritty; themes:grief; emotional_arc:tragic/unsettling,none,0.178773,0.0,0.213176,0.669990,0.349560,0.5,0.5,71.7,0.835404,"The Second Best Exotic Marigold Hotel is a 2015 comedy drama film that is the sequel to The Best Exotic Marigold Hotel. The motion picture starred ensemble cast consisting of Judi Dench, Maggie Smith, Dev Patel, Bill..."
4859,tt5726616,Call Me by Your Name,2017,movie,"Drama,Romance","Romance, Relationships, and Romantic Comedy",direct_request,something beautiful human a little emotional but please do not destroy me,something beautiful human a little emotional but please do not destroy me,none,none,mood:dark/sad; tone:bleak/gritty; themes:grief; emotional_arc:tragic/unsettling,none,0.480220,0.0,0.261138,0.754063,0.349560,0.5,0.5,74.6,0.826014,"Call Me by Your Name is a 2017 coming-of-age romantic drama film directed by Luca Guadagnino. Its screenplay, by James Ivory, who also co-produced, is based on the 2007 novel by André Aciman. The film is the final in..."
3099,tt1645080,The Art of Getting By,2011,movie,"Drama,Romance","Romance, Relationships, and Romantic Comedy",direct_request,something beautiful human a little emotional but please do not destroy me,something beautiful human a little emotional but please do not destroy me,none,none,mood:dark/sad; tone:bleak/gritty; themes:grief; emotional_arc:tragic/unsettling,none,0.344686,0

Saved prompt evaluation: data/prompt_recommendation_evaluation.csv
Saved prompt stress summary: data/prompt_recommendation_stress_summary.csv


,category,prompt,result_count,top_match_score,score_band,warning,top_title,top_genres,inferred_genres,excluded_genres,excluded_facets
0,clear genre mood,"I want something exciting, futuristic, and intense",10,90.7,strong,ok,Tenet,"Action,Sci-Fi,Thriller","Action, Sci-Fi, Thriller",none,none
1,clear comfort,"I want a comforting, funny, wholesome family movie",10,85.4,strong,ok,Paddington 2,"Adventure,Comedy,Family","Family, Comedy",none,none
2,dark specific,Give me a dark psychological and mind-bending thriller,10,96.5,strong,ok,Saw,"Horror,Mystery,Thriller","Mystery, Thriller",none,none
3,romantic mixed emotion,"I want a romantic, bittersweet musical with strong emotions",10,85.7,strong,ok,Mohabbatein,"Drama,Musical,Romance","Musical, Romance",none,none
4,specific horror,I want a tense zombie survival horror movie,10,92.8,strong,ok,REC,"Horror,Mystery,Thriller","Horror, Thriller",none,none
5,current feeling only,I feel sad and heartbroken,10,76.7,usable,ok,Raising Hope,"Comedy,Family","Comedy, Family",none,mood:dark/sad; tone:bleak/gritty; themes:grief/revenge; emotional_arc:tragic/unsettling
6,current feeling plus desire,I feel sad and heartbroken and I want to watch something joyous and comedic,10,96.1,strong,ok,Chappelle's Show,"Comedy,Music",Comedy,none,none
7,negation,I want something funny but not scary,10,95.5,strong,ok,Chappelle's Show,"Comedy,Music",Comedy,Horror,mood:dark/tense; tone:bleak/suspenseful; emotional_arc:unsettling
8,typos negation,"I dont want sad movie, need joyful commedy",10,88.8,strong,ok,Yes Man,"Comedy,Romance",Comedy,none,mood:sad; themes:grief; emotional_arc:tragic
9,broken english,me sad heart broken want watch happy funny,10,93.3,strong,ok,QI,"Comedy,Game-Show",Comedy,none,none


## Phase 8: Evaluation Seeds And Prompt Stress Tests

This section validates both recommendation surfaces used by the project. The title-to-title tests check whether a known movie or show returns semantically related results. The free-text prompt tests are deliberately harder: they include contradictions, misspellings, imperfect English, current emotional state, explicit desired mood, and negative constraints.

The stress-test prompts are not only happy-path examples. They include cases such as `I feel sad and heartbroken`, `I want something funny but not scary`, `I dont want sad movie, need joyful commedy`, and longer requests with multiple exclusions. These are important because a mood recommender should not simply match isolated words. It should understand when a sad word describes the user, when it describes the desired content, and when the user is explicitly asking to avoid that feeling.

Two validation files are produced:

- `data/prompt_recommendation_evaluation.csv` stores every returned recommendation with match scores, inferred genres, exclusions, and negative-match signals.
- `data/prompt_recommendation_stress_summary.csv` stores one summary row per prompt, including the top score, score band, warning text, and top recommendation.

Score bands are interpreted conservatively: `strong` means the top match is very defensible, `usable` means the result is acceptable but should still be read by a human, `borderline` means the prompt is difficult or underspecified, and `weak` means the prompt needs better parsing, more labeled training examples, or a richer catalog.


In [12]:
evaluation_titles = [
    "Parasite",
    "Inside Out",
    "The Pianist",
    "Train to Busan",
    "The Matrix",
    "Her",
    "Get Out",
    "La La Land",
]

for title in evaluation_titles:
    print("\n" + "=" * 100)
    print(f"NLP recommendations for: {title}")
    try:
        display(recommend_similar_nlp(title, top_n=10, min_votes=30_000))
    except Exception as exc:
        print("ERROR:", exc)



NLP recommendations for: Parasite


,tconst,primary_title,release_year,content_type,genres,bertopic_topic,bertopic_topic_label,bertopic_low_confidence,semantic_similarity,facet_similarity,topic_similarity,genre_similarity,scoring_profile,match_score,final_score,description
5284,tt7282468,Burning,2018,movie,"Drama,Mystery,Thriller",13,"East Asian Cinema, Anime, and Serialized Action",False,0.544144,0.823513,1.000000,0.666667,balanced,84.2,0.842468,"Burning (Korean: 버닝) is a 2018 South Korean psychological thriller film co-written, produced, and directed by Lee Chang-dong. The film is based on the short story ""Barn Burning"" from The Elephant Vanishes by Haruki M..."
2210,tt1216496,Mother,2009,movie,"Crime,Drama,Mystery",13,"East Asian Cinema, Anime, and Serialized Action",False,0.594153,0.871949,1.000000,0.250000,balanced,82.0,0.819691,"Mother (Korean: 마더) is a 2009 South Korean neo-noir psychological thriller film directed by Bong Joon Ho, starring Kim Hye-ja and Won Bin. The plot follows a mother who, after her intellectually disabled son is accus..."
661,tt0260991,Joint Security Area,2000,movie,"Action,Drama,Thriller",13,"East Asian Cinema, Anime, and Serialized Action",False,0.468333,0.863648,1.000000,0.666667,balanced,80.5,0.804605,"Joint Security Area is a 2000 South Korean mystery thriller film directed and co-written by Park Chan-wook and based on the novel DMZ by Park Sang-yeon. It is Park Chan-wook's third film as director, and is frequentl..."
2154,tt1190539,The Chaser,2008,movie,"Action,Crime,Drama",13,"East Asian Cinema, Anime, and Serialized Action",False,0.575877,0.798353,1.000000,0.250000,balanced,79.1,0.790654,The Chaser (Korean: 추격자) is a 2008 South Korean action thriller film starring Kim Yoon-seok and Ha Jung-woo. It was directed by Na Hong-jin in his directorial debut. Inspired by real-life Korean serial killer Yoo You...
937,tt0364569,Oldboy,2003,movie,"Action,Drama,Mystery",13,"East Asian Cinema, Anime, and Serialized Action",False,0.524725,0.881217,1.000000,0.250000,balanced,78.3,0.783219,"Oldboy (Korean: 올드보이) is a 2003 South Korean action thriller film directed and co-written by Park Chan-wook. A loose adaptation of the Japanese manga Old Boy by Garon Tsuchiya and Nobuaki Minegishi, the film follows ..."
4690,tt5215952,The Wailing,2016,movie,"Drama,Horror,Mystery",13,"East Asian Cinema, Anime, and Serialized Action",False,0.539163,0.919251,1.000000,0.250000,balanced,78.1,0.780618,"The Wailing is a 2016 South Korean horror film written and directed by Na Hong-jin and starring Kwak Do-won, Hwang Jung-min and Chun Woo-hee. The film centers on a policeman who investigates a series of mysterious ki..."
1296,tt0468492,The Host,2006,movie,"Drama,Horror,Sci-Fi",13,"East Asian Cinema, Anime, and Serialized Action",False,0.623287,0.868499,1.000000,0.250000,balanced,78.0,0.780007,The Host is a 2006 monster film directed and co-written by Bong Joon Ho. It stars Song Kang-ho as food stand vendor Park Gang-du whose daughter Hyun-seo is kidnapped by a creature dwelling around the Han River in Seo...
1137,tt0423866,3-Iron,2004,movie,"Crime,Drama,Romance",13,"East Asian Cinema, Anime, and Serialized Action",False,0.522771,0.858738,1.000000,0.250000,balanced,77.0,0.770357,"Tae-suk (Jae Hee) is a loner who drives around on his motorbike, taping takeout menus over the keyholes of front doors and breaking into apartments where the menus have not been removed. He lives in those apartments ..."
2566,tt1379182,Dogtooth,2009,movie,"Drama,Thriller",3,"Thriller, Crime, and High-Stakes Action",False,0.434407,0.871451,0.714941,1.000000,balanced,76.5,0.764833,"A controlling, manipulative father (Christos Stergioglou) locks his three adult offspring in a state of perpetual childhood by keeping them prisoner within the sprawling family compound. The children are bored to tea..."
944,tt0365737,Syriana,2005,movie,"Drama,Thriller",5,"Horror, Supernatural Threats, and Dark Suspense",False,0.443873,0.881357,0.682440,1.000000,balanced,76.0,0.760407,"Syriana is a 2005 Americ


NLP recommendations for: Inside Out


,tconst,primary_title,release_year,content_type,genres,bertopic_topic,bertopic_topic_label,bertopic_low_confidence,semantic_similarity,facet_similarity,topic_similarity,genre_similarity,scoring_profile,match_score,final_score,description
215,tt2948372,Soul,2020,movie,"Adventure,Animation,Comedy",15,"Music, Performance, and Artist Journeys",False,0.185836,0.742825,0.826548,1.0,animation_family,70.9,0.708902,"Soul is a 2020 American animated fantasy comedy-drama film produced by Pixar Animation Studios for Walt Disney Pictures. Directed by Pete Docter, who co-wrote it with Mike Jones and Kemp Powers, the film stars the vo..."
2429,tt1323594,Despicable Me,2010,movie,"Adventure,Animation,Comedy",10,"Animation, Superheroes, and Family Adventure",False,0.200370,0.691249,0.734632,1.0,animation_family,68.6,0.685718,"Despicable Me is a 2010 American animated comedy film directed by Chris Renaud and Pierre Coffin and written by Cinco Paul and Ken Daurio. The first feature film produced by Illumination Entertainment, it features th..."
4652,tt5104604,Isle of Dogs,2018,movie,"Adventure,Animation,Comedy",10,"Animation, Superheroes, and Family Adventure",False,0.221185,0.628665,0.734632,1.0,animation_family,68.2,0.681850,"Isle of Dogs is a 2018 adult stop-motion animated science fiction comedy-drama film written, produced, and directed by Wes Anderson, narrated by Courtney B. Vance and starring an ensemble cast consisting of Bryan Cra..."
1452,tt0808506,The Girl Who Leapt Through Time,2006,movie,"Adventure,Animation,Comedy",13,"East Asian Cinema, Anime, and Serialized Action",True,0.280110,0.707569,0.678992,1.0,animation_family,66.1,0.661488,"At Kuranose High School in Tokyo, Japan, 17-year-old Makoto Konno discovers a message written on a blackboard that reads ""Time waits for no one"" and ends up inadvertently falling onto a walnut-shaped object. On her w..."
1031,tt0388473,Tokyo Godfathers,2003,movie,"Adventure,Animation,Comedy",13,"East Asian Cinema, Anime, and Serialized Action",False,0.199826,0.627550,0.678992,1.0,animation_family,66.1,0.660550,"Tokyo Godfathers is a 2003 Japanese animated Christmas tragicomedy adventure film written and directed by Satoshi Kon. The film stars live-action actors such as Toru Emori, Yoshiaki Umegaki, and Aya Okamoto as the le..."
3674,tt2294629,Frozen,2013,movie,"Adventure,Animation,Comedy",18,"Holiday, Family, and Comfort Viewing",False,0.198157,0.575277,0.770303,1.0,animation_family,65.3,0.652764,"Frozen is a 2013 American animated musical fantasy film produced by Walt Disney Animation Studios, loosely based on Hans Christian Andersen's 1844 fairy tale ""The Snow Queen"". Directed by Chris Buck and Jennifer Lee ..."
4533,tt4729430,Klaus,2019,movie,"Adventure,Animation,Comedy",18,"Holiday, Family, and Comfort Viewing",False,0.169921,0.566892,0.770303,1.0,animation_family,64.9,0.649233,"Klaus is a 2019 animated Christmas adventure comedy film co-written, co-produced, and directed by Sergio Pablos in his directorial debut, produced by his company the SPA Studios and distributed by Netflix. A traditio..."
2332,tt12801262,Luca,2021,movie,"Adventure,Animation,Comedy",12,International and Auteur Drama,False,0.166120,0.573781,0.850105,1.0,animation_family,64.8,0.648270,Luca is a 2021 American animated coming-of-age fantasy comedy film produced by Pixar Animation Studios for Walt Disney Pictures. The film was directed by Enrico Casarosa and written by Jesse Andrews and Mike Jones fr...
5036,tt6324278,Abominable,2019,movie,"Adventure,Animation,Comedy",10,"Animation, Superheroes, and Family Adventure",False,0.229493,0.589107,0.734632,1.0,animation_family,64.6,0.645818,"Abominable is a 2019 animated fantasy adventure comedy film produced by DreamWorks Animation and Pearl Studio. Written and directed by Jill Culton, the film stars the voices of Chloe Bennet, Albert Tsai, Tenzing Norg..."
4172,tt3521164,Moana,2016,movie,"Adventure,Animation,Comedy",10,"Animation, Superheroes, and Family Adventure",False,0.168363,0


NLP recommendations for: The Pianist


...

,tconst,primary_title,release_year,content_type,genres,bertopic_topic,bertopic_topic_label,bertopic_low_confidence,semantic_similarity,facet_similarity,topic_similarity,genre_similarity,scoring_profile,match_score,final_score,description
736,tt0287467,Talk to Her,2002,movie,"Drama,Mystery,Romance",2,"Romance, Relationships, and Romantic Comedy",False,0.565030,0.907337,1.000000,0.500000,balanced,85.8,0.857940,"Talk To Her is a 2002 Spanish psychological melodrama film, written and directed by Pedro Almodóvar. It stars: Javier Cámara, Darío Grandinetti, Leonor Watling, Geraldine Chaplin, and Rosario Flores. The film follows..."
854,tt0332280,The Notebook,2004,movie,"Drama,Romance",2,"Romance, Relationships, and Romantic Comedy",False,0.493314,0.878484,1.000000,0.666667,balanced,83.3,0.832525,"The Notebook is a 2004 American romantic drama film directed by Nick Cassavetes, from a screenplay by Jeremy Leven and Jan Sardi, and based on the 1996 novel by Nicholas Sparks. The film stars Ryan Gosling and Rachel..."
5,tt0120338,Titanic,1997,movie,"Drama,Romance",2,"Romance, Relationships, and Romantic Comedy",False,0.464861,0.915849,1.000000,0.666667,balanced,82.4,0.823853,"Titanic is a 1997 American epic historical romance film written and directed by James Cameron. Incorporating both historical and fictional aspects, it is based on accounts of the sinking of RMS Titanic in 1912. Leona..."
2932,tt1549572,Another Earth,2011,movie,"Drama,Romance,Sci-Fi",2,"Romance, Relationships, and Romantic Comedy",False,0.449162,0.909468,1.000000,1.000000,balanced,81.9,0.819380,"Another Earth is a 2011 American science fiction drama film directed by Mike Cahill and starring Brit Marling and William Mapother. It premiered at the 2011 Sundance Film Festival in January, and was given a limited ..."
3747,tt2381111,Brooklyn,2015,movie,"Drama,Romance",8,"Thriller, Crime, and High-Stakes Action",False,0.527315,0.867764,0.809958,0.666667,balanced,81.5,0.815373,"Brooklyn is a 2015 romantic period drama film directed by John Crowley and written by Nick Hornby, based on the 2009 novel by Colm Tóibín. It stars Saoirse Ronan in the lead role, with Emory Cohen, Domhnall Gleeson, ..."
407,tt0109830,Forrest Gump,1994,movie,"Drama,Romance",2,"Romance, Relationships, and Romantic Comedy",False,0.459466,0.860023,1.000000,0.666667,balanced,80.9,0.809259,"Slow-witted Forrest Gump (Tom Hanks) has never thought of himself as disadvantaged, and thanks to his supportive mother (Sally Field), he leads anything but a restricted life. Whether dominating on the gridiron as a ..."
3387,tt1937264,Now Is Good,2012,movie,"Drama,Romance",6,"Romance, Relationships, and Romantic Comedy",False,0.534233,0.892335,0.865638,0.666667,balanced,79.8,0.798346,"Tessa (Dakota Fanning) makes a list of things she wants to do before she passes away from leukemia. Topping the list is her desire to lose her virginity. Now Is Good is a 2012 movie classified within Drama, Romance. ..."
1087,tt0408777,The Edukators,2004,movie,"Drama,Romance",2,"Romance, Relationships, and Romantic Comedy",False,0.462904,0.897859,1.000000,0.666667,balanced,79.7,0.797050,"Peter (Stipe Erceg) and Jan (Daniel Brühl) are ""edukators,"" anarchists who break into wealthy people's homes -- never stealing, but serving notice to fatcats that their days are numbered. Along comes Jule (Julia Jent..."
3499,tt2082197,Barfi!,2012,movie,"Comedy,Drama,Romance",6,"Romance, Relationships, and Romantic Comedy",False,0.511838,0.871834,0.865638,0.500000,balanced,79.4,0.793909,"Shruti loves Barfi, a hearing and speech-impaired man, but marries someone else. Years later, she learns that he is in love with an autistic girl, and feels the need to rethink her own marriage. is a 2012 movie class..."
2515,tt1355644,Passengers,2016,movie,"Drama,Romance,Sci-Fi",2,"Romance, Relationships, and Romantic Comedy",False,0.430681,0.847877,1.000000,1.000000,balanced,79.1,0.790773,"Passengers is a 2016 American science fiction romance film directed by Morten T


NLP recommendations for: Get Out


,tconst,primary_title,release_year,content_type,genres,bertopic_topic,bertopic_topic_label,bertopic_low_confidence,semantic_similarity,facet_similarity,topic_similarity,genre_similarity,scoring_profile,match_score,final_score,description
3012,tt1591095,Insidious,2010,movie,"Horror,Mystery,Thriller",5,"Horror, Supernatural Threats, and Dark Suspense",False,0.572569,0.924946,1.0,1.000000,balanced,90.4,0.904246,"Insidious is a 2010 supernatural horror film directed and co-edited by James Wan, written by Leigh Whannell, and starring Patrick Wilson, Rose Byrne, and Barbara Hershey. It is the first installment in the Insidious ..."
1025,tt0387564,Saw,2004,movie,"Horror,Mystery,Thriller",5,"Horror, Supernatural Threats, and Dark Suspense",False,0.511872,0.858551,1.0,1.000000,balanced,89.6,0.896267,"Saw is a 2004 American horror film directed by James Wan in his feature directorial debut, and written by Leigh Whannell, from a story by Wan and Whannell. It stars Whannell alongside Cary Elwes, Danny Glover, Monica..."
4081,tt3235888,It Follows,2014,movie,"Horror,Mystery,Thriller",5,"Horror, Supernatural Threats, and Dark Suspense",False,0.554033,0.895703,1.0,1.000000,balanced,88.5,0.884507,"It Follows is a 2014 American supernatural psychological horror film written and directed by David Robert Mitchell. It stars Maika Monroe as a young woman who is pursued by a supernatural entity. Keir Gilchrist, Dani..."
265,tt4972582,Split,2016,movie,"Horror,Thriller",5,"Horror, Supernatural Threats, and Dark Suspense",False,0.598600,0.936605,1.0,0.666667,balanced,88.1,0.880745,"Split is a 2016 American psychological thriller film written, directed and produced by M. Night Shyamalan, and starring James McAvoy, Anya Taylor-Joy, and Betty Buckley. It is the second installment in the Unbreakabl..."
3613,tt2226417,Insidious: Chapter 2,2013,movie,"Horror,Mystery,Thriller",5,"Horror, Supernatural Threats, and Dark Suspense",False,0.539432,0.912376,1.0,1.000000,balanced,87.7,0.877236,"Insidious: Chapter 2 is a 2013 American supernatural horror film directed by James Wan. It is the sequel to Insidious (2010), and the second installment in the Insidious franchise, and the fourth in terms of the seri..."
89,tt1259521,The Cabin in the Woods,2011,movie,"Horror,Mystery,Thriller",5,"Horror, Supernatural Threats, and Dark Suspense",False,0.493150,0.966795,1.0,1.000000,balanced,86.3,0.862944,"The Cabin in the Woods is a 2011 science fiction comedy horror film directed by Drew Goddard in his directorial debut, produced by Joss Whedon, and written by Whedon and Goddard. It stars Kristen Connolly, Chris Hems..."
1737,tt1038988,REC,2007,movie,"Horror,Mystery,Thriller",5,"Horror, Supernatural Threats, and Dark Suspense",False,0.467480,0.888407,1.0,1.000000,balanced,86.1,0.860566,"Rec is a 2007 Spanish found footage zombie horror film co-written and directed by Jaume Balagueró and Paco Plaza. The film stars Manuela Velasco as television reporter Ángela Vidal, who, along with her cameraman, acc..."
298,tt6857112,Us,2019,movie,"Horror,Mystery,Thriller",5,"Horror, Supernatural Threats, and Dark Suspense",False,0.515931,0.892834,1.0,1.000000,balanced,85.8,0.857720,"Us is a 2019 American psychological horror film written and directed by Jordan Peele, and starring Lupita Nyong'o, Winston Duke, Elisabeth Moss and Tim Heidecker. The story follows Adelaide Wilson and her family, who..."
3374,tt1922777,Sinister,2012,movie,"Horror,Mystery,Thriller",5,"Horror, Supernatural Threats, and Dark Suspense",False,0.528347,0.853884,1.0,1.000000,balanced,85.7,0.856548,"Sinister is a 2012 supernatural horror film directed by Scott Derrickson, who co-wrote with C. Robert Cargill. It stars Ethan Hawke as a true-crime writer whose discovery of snuff films depicting gruesome murders and..."
2982,tt15791034,Barbarian,2022,movie,"Horror,Mystery,Thriller",5,"Horror, Supernatural Threats, and Dark Suspense",False,0.493365,0.921532,1.0,1.000000,balanced,85.3,0.853223,"Barbarian is a 2022 American horr


NLP recommendations for: La La Land


,tconst,primary_title,release_year,content_type,genres,bertopic_topic,bertopic_topic_label,bertopic_low_confidence,semantic_similarity,facet_similarity,topic_similarity,genre_similarity,scoring_profile,match_score,final_score,description
7408,tt3544112,Sing Street,2016,movie,"Comedy,Drama,Music",2,"Romance, Relationships, and Romantic Comedy",True,0.457891,0.922592,0.826548,1.000000,balanced,78.4,0.784359,"Sing Street is a 2016 coming-of-age musical comedy-drama film written and directed by John Carney from a story by Carney and Simon Carmody. Starring Lucy Boynton, Maria Doyle Kennedy, Aidan Gillen, Jack Reynor, Kelly..."
5237,tt7131622,Once Upon a Time in Hollywood,2019,movie,"Comedy,Drama",8,"Thriller, Crime, and High-Stakes Action",False,0.443127,0.905621,0.843393,0.666667,balanced,77.8,0.778198,"Once Upon a Time. in Hollywood is a 2019 comedy-drama film written and directed by Quentin Tarantino. Produced by Columbia Pictures in association with Bona Film Group, Heyday Films, and Visiona Romantica, and distri..."
6491,tt0450188,La Vie En Rose,2007,movie,"Biography,Drama,Music",12,International and Auteur Drama,False,0.462975,0.894633,0.818004,0.500000,balanced,75.2,0.752368,"La Vie en Rose is a 2007 biographical musical film about the life of French singer Édith Piaf, co-written and directed by Olivier Dahan, and starring Marion Cotillard as Piaf. The UK and US title La Vie en Rose comes..."
3428,tt1980929,Begin Again,2013,movie,"Comedy,Drama,Music",2,"Romance, Relationships, and Romantic Comedy",False,0.336457,0.914997,0.826548,1.000000,balanced,75.2,0.751878,"Begin Again is a 2013 American musical comedy-drama film written and directed by John Carney, and starring Keira Knightley and Mark Ruffalo. Knightley plays a singer-songwriter who is discovered by a struggling recor..."
1153,tt0426931,August Rush,2007,movie,"Drama,Music",15,"Music, Performance, and Artist Journeys",False,0.400351,0.856222,1.000000,0.666667,balanced,74.6,0.746108,"In 1995, Lyla Novacek is a cellist studying at the Juilliard School. Louis Connelly is the lead singer of an Irish rock band. They meet and have a one-night stand but are unable to maintain contact. Lyla discovers th..."
5701,tt8721424,"tick, tick... BOOM!",2021,movie,"Biography,Comedy,Drama",15,"Music, Performance, and Artist Journeys",False,0.398816,0.917782,1.000000,0.500000,balanced,73.8,0.738454,"Tick, Tick. Boom! is a 2021 American biographical musical film directed by Lin-Manuel Miranda in his feature directorial debut. Written by Steven Levenson, who also serves as an executive producer, it is based on the..."
4810,tt5613484,Mid90s,2018,movie,"Comedy,Drama",8,"Thriller, Crime, and High-Stakes Action",False,0.411649,0.913581,0.843393,0.666667,balanced,73.0,0.730022,"Mid90s is a 2018 American coming-of-age comedy-drama film written and directed by Jonah Hill, in his feature directorial debut. The plot follows Stevie, a 13-year-old boy in 1990s Los Angeles. To escape a troubled ho..."
4098,tt3312830,Youth,2015,movie,"Comedy,Drama,Music",12,International and Auteur Drama,False,0.353025,0.840936,0.818004,1.000000,balanced,72.8,0.727709,"At a luxury Swiss Alps resort, retired composer Fred Ballinger and filmmaker Mick Boyle reflect on aging, memory, art, desire, regret, and the fragile pull of the past. Youth is a 2015 movie classified within Comedy,..."
5521,tt8079248,Yesterday,2019,movie,"Comedy,Fantasy,Music",15,"Music, Performance, and Artist Journeys",False,0.436766,0.902308,1.000000,0.500000,balanced,71.7,0.717021,Yesterday is a 2019 romantic comedy film directed by Danny Boyle and written by Richard Curtis based on a story by Jack Barth and Curtis. It stars Himesh Patel as a struggling musician who enters an alternative prese...
130,tt1517451,A Star Is Born,2018,movie,"Drama,Music,Romance",15,"Music, Performance, and Artist Journeys",False,0.360965,0.902947,1.000000,0.500000,balanced,71.5,0.714657,"43-year-old Jackson ""Jack"" Maine, a country rock singer privately battling alcoho

## Phase 9: Final Model Review Notes

This notebook produces the final files needed to inspect and defend the recommender:

- `data/bertopic_topic_info.csv`: topic keywords plus human-readable topic labels and quality flags.
- `data/bertopic_topic_review.csv`: compact topic review table with dominant genres, sample titles, confidence, and watchlist flags.
- `data/bertopic_topic_samples.csv`: representative titles for every topic.
- `data/title_facet_labels.csv`: top mood, tone, theme, conflict, pace, emotional arc, and setting labels for every title.
- `data/final_nlp_recommender_features.csv`: final model-ready feature table used to build the deployable catalog.
- `data/prompt_recommendation_evaluation.csv`: detailed stress-test results for current emotion, desired mood, exclusions, typos, and imperfect English.
- `data/prompt_recommendation_stress_summary.csv`: compact stress-test summary with score bands and warnings.

Final review guidance:

1. Inspect any topic marked `watchlist` or `needs_review` in `data/bertopic_topic_review.csv`. MiniBatchKMeans reduces hard BERTopic outliers, so this project tracks low-confidence assignments instead of pretending every assignment is equally strong.
2. Use the stress summary to identify prompts with `borderline`, `weak`, or warning statuses. These are the safest prompts to discuss as limitations because they reveal where natural language is genuinely ambiguous.
3. Keep prompt diagnostics visible. The output columns for inferred genres, excluded genres, excluded facets, and negative-match signals show that the system is doing more than simple keyword matching.
4. Interpret match percentages as calibrated decision support, not objective truth. A high score means the title matches the model's interpreted request; the final product would still benefit from user feedback buttons and a manually labeled relevance benchmark.
5. The strongest future improvement would be collecting 50-100 labeled prompts with expected good and bad recommendations, then tuning ranking weights against nDCG or top-k relevance. For this submission, the notebook includes a practical stress-test set and the backend adds production rules for harder real-user phrasing.

Connection to the deployed product: the website does not run this notebook at request time. The notebook explains and produces the modeling artifacts; `backend/build_catalog.py` converts the final feature table into `assets/data/catalog.json`; the FastAPI backend loads that catalog and serves both the frontend and recommendation API on Render.
